# Task 1 
# Regresi estimasi waktu pengiriman paket
**Muhammad Habibie Rabbani · 101032300005 · Kelompok 11**

Seluruh kode audit, pelatihan, kalibrasi, evaluasi, interpretasi, dan prediksi ada di notebook ini. Tidak mengimpor atau menjalankan skrip proyek `.py`.

Pilih kernel **.venv**, lalu **Run All**. Bisa juga klik ▶ per cell dari atas ke bawah. Setiap tahap memiliki cell definisi fungsi dan cell menjalankan tahap. Tunggu satu cell selesai sebelum lanjut.

P50 adalah perkiraan tengah; P90 adalah batas waktu yang ditargetkan mencakup sekitar 90% paket. Bukan jaminan setiap paket tiba tepat waktu.

Run All menghitung ulang dan menimpa hasil, sekitar 8 menit pada laptop ini. Dataset mentah tidak diubah. Tidak ada commit/push otomatis. Hasil tugas lain belum termasuk.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json, sys, time, hashlib
try:
    from IPython.display import display, Markdown, Image
except ImportError:
    # Peluncur terminal hanya memerlukan requirements-task1.txt.
    def display(value): print(value)
    def Markdown(value): return value
    def Image(filename=None, **kwargs): return filename

# PROJECT_DIR boleh diisi dengan folder repository; DATA_DIR dengan folder dataset.
start_dir = Path(globals().get("PROJECT_DIR", Path.cwd())).resolve()
candidates = [start_dir, *start_dir.parents, start_dir / "UTS-ML-KELOMPOK-11"]
REPO = next((d for d in candidates if (d / "kode/Task1_Regresi.ipynb").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Buka notebook dari folder kode atau root repository, atau isi PROJECT_DIR.")
DATA_DIR = Path(globals().get("DATA_DIR", REPO.parent)).resolve()
(REPO / "kode/hasil").mkdir(parents=True, exist_ok=True)
print("Python:", sys.executable)
print("Folder hasil:", REPO / "kode/hasil")
print("Folder data:", DATA_DIR)

missing = [n for n in ['regresi_train.csv.gz', 'regresi_test.csv.gz'] if not (DATA_DIR / n).is_file()]
if missing and globals().get('RUN_MODE', 'full') == 'full':
    raise FileNotFoundError(f"Dataset tidak ditemukan di {DATA_DIR}: {missing}. Isi DATA_DIR pada cell ini atau gunakan --data-dir.")


## 1. Audit awal
Periksa jumlah baris, tanggal, nilai kosong, dan ID ganda.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Audit deskriptif awal; tidak mengubah data atau melatih model."""
from pathlib import Path
import numpy as np
import pandas as pd

def audit(path: Path, train: bool) -> set:
    data = pd.read_csv(path)
    print(f'\n=== {path.name} ===')
    print(f'Baris: {len(data):,} | Kolom: {len(data.columns)}')
    print(f'ID kosong: {data['id_paket'].isna().sum():,}')
    print(f'Duplikat ID (kemunculan setelah pertama): {data['id_paket'].dropna().duplicated().sum():,}')
    print(f'Duplikat seluruh baris (setelah pertama): {data.duplicated().sum():,}')
    tanggal = pd.to_datetime(data['tgl_pickup'], errors='coerce')
    print(f'Rentang pickup: {tanggal.min()} sampai {tanggal.max()}')
    print(f'Tanggal kosong/tidak terbaca: {tanggal.isna().sum():,}')
    print('\nJumlah paket per bulan:')
    print(tanggal.dt.to_period('M').value_counts().sort_index().to_string())
    missing = data.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    print('\nKolom dengan nilai kosong:')
    if missing.empty:
        print('Tidak ditemukan oleh pembaca CSV.')
    else:
        print(pd.DataFrame({'jumlah': missing, 'persen': (100 * missing / len(data)).round(2)}).to_string())
    if train:
        target = pd.to_numeric(data['durasi_jam'], errors='coerce')
        finite = target.notna() & np.isfinite(target)
        print('\nPemeriksaan durasi_jam:')
        print(f'Kosong/non-numerik: {target.isna().sum():,}')
        print(f'Tak hingga: {np.isinf(target).sum():,}')
        print(f'Nol/negatif: {(target[finite] <= 0).sum():,}')
        print(target[finite].describe(percentiles=[0.01, 0.5, 0.9, 0.99]).to_string())
    else:
        print(f'\nKolom target ada di test: {'durasi_jam' in data.columns}')
    return set(data['id_paket'].dropna())

def jalankan_audit_awal_regresi() -> None:
    args = SimpleNamespace(data_dir=DATA_DIR)
    paths = [args.data_dir / name for name in ('regresi_train.csv.gz', 'regresi_test.csv.gz')]
    for path in paths:
        if not path.is_file():
            raise FileNotFoundError(f'Dataset tidak ditemukan: {path}')
    train_ids = audit(paths[0], train=True)
    test_ids = audit(paths[1], train=False)
    print(f'\nID yang muncul di train dan test: {len(train_ids & test_ids):,}')
    print('\nAudit awal selesai. Belum ada data yang dibuang atau diubah.')
    print('Audit leakage, kategori, dan outlier perlu pemeriksaan lanjutan.')

In [ ]:
jalankan_audit_awal_regresi()

## 2. Audit lanjutan
Pelajari pola nilai kosong, kategori, dan nilai ekstrem tanpa mengubah data mentah.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Audit deskriptif Task 1. Menyimpan ringkasan, tanpa mengubah dataset."""
import json
import platform
from importlib.metadata import version
from pathlib import Path
import numpy as np
import pandas as pd
KATEGORI_audit_lanjutan_regresi = ['layanan', 'mitra_kurir', 'hub_asal', 'kota_asal', 'pulau_tujuan', 'kota_tujuan']

def normalisasi_audit_lanjutan_regresi(series):
    return series.astype('string').str.strip().str.lower()

def ringkasan_dataset(data):
    numerik = data.select_dtypes(include='number')
    aturan = {'berat_kg_nol_negatif': data.berat_kg <= 0, 'jarak_km_negatif': data.jarak_km < 0, 'volume_liter_nol_negatif': data.volume_liter <= 0, 'jumlah_item_bukan_bilangan_bulat_positif': (data.jumlah_item <= 0) | (data.jumlah_item % 1 != 0), 'nilai_barang_negatif': data.nilai_barang < 0, 'alamat_skor_di_luar_0_1': ~data.alamat_skor.between(0, 1), 'indeks_kemacetan_di_luar_0_1': ~data.indeks_kemacetan.between(0, 1), 'jam_pickup_tidak_valid': ~data.jam_pickup.between(0, 23) | (data.jam_pickup % 1 != 0), 'asuransi_bukan_0_1': ~data.asuransi.isin([0, 1]), 'cod_bukan_0_1': ~data.cod.isin([0, 1]), 'hujan_negatif': (data.hujan_asal_mm < 0) | (data.hujan_tujuan_mm < 0)}
    kategori = {}
    for kolom in KATEGORI_audit_lanjutan_regresi:
        asli = data[kolom].astype('string')
        bersih = normalisasi_audit_lanjutan_regresi(asli)
        kategori[kolom] = {'unik_asli': int(asli.nunique()), 'unik_setelah_normalisasi': int(bersih.nunique()), 'baris_berubah_penulisan': int((asli != bersih).sum())}
    return {'baris': len(data), 'nilai_kosong': {k: int(v) for k, v in data.isna().sum().items() if v}, 'nilai_tak_hingga': {k: int(v) for k, v in np.isinf(numerik).sum().items() if v}, 'pelanggaran_aturan': {k: int(v.sum()) for k, v in aturan.items()}, 'kategori': kategori}

def jalankan_audit_lanjutan_regresi():
    repo = (REPO / 'kode/notebook.py').parents[1]
    args = SimpleNamespace(data_dir=DATA_DIR, output_dir=repo / 'kode' / 'hasil' / 'audit_regresi')
    train = pd.read_csv(args.data_dir / 'regresi_train.csv.gz')
    test = pd.read_csv(args.data_dir / 'regresi_test.csv.gz')
    hasil = {'versi': {'python': platform.python_version(), **{paket: version(paket) for paket in ['pandas', 'numpy', 'scipy']}}, 'train': ringkasan_dataset(train), 'test': ringkasan_dataset(test)}
    hasil['target'] = {'kuantil': {str(q): float(train.durasi_jam.quantile(q)) for q in [0, 0.5, 0.9, 0.99, 0.999, 1]}, 'jumlah_di_atas_jam': {str(t): int((train.durasi_jam > t).sum()) for t in [168, 720, 2160]}}
    pickup = pd.to_datetime(train.tgl_pickup, errors='coerce') + pd.to_timedelta(train.jam_pickup, unit='h')
    tiba = pickup + pd.to_timedelta(train.durasi_jam, unit='h')
    hasil['konsistensi_snapshot'] = {'asumsi_batas_eksklusif': '2025-09-01 00:00:00', 'perkiraan_tiba_setelah_snapshot': int((tiba >= pd.Timestamp('2025-09-01')).sum())}
    unseen = {}
    for kolom in KATEGORI_audit_lanjutan_regresi:
        train_values = set(normalisasi_audit_lanjutan_regresi(train[kolom]).dropna())
        test_values = normalisasi_audit_lanjutan_regresi(test[kolom])
        baru = test_values.notna() & ~test_values.isin(train_values)
        unseen[kolom] = {'kategori_baru': int(test_values[baru].nunique()), 'baris_test_terdampak': int(baru.sum())}
    hasil['kategori_test_baru_setelah_normalisasi'] = unseen
    corr = train.select_dtypes(include='number').corrwith(train.durasi_jam, method='spearman')
    corr = corr.drop('durasi_jam').dropna()
    corr = corr.loc[corr.abs().sort_values(ascending=False).index]
    args.output_dir.mkdir(parents=True, exist_ok=True)
    (args.output_dir / 'ringkasan.json').write_text(json.dumps(hasil, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    corr.rename('spearman_durasi_jam').rename_axis('fitur').to_csv(args.output_dir / 'korelasi_target.csv')
    for nama, data in [('train', train), ('test', test)]:
        audit = data.assign(bulan=pd.to_datetime(data.tgl_pickup).dt.strftime('%Y-%m'), berat_kosong=data.berat_kg.isna())
        for grup in ['bulan', 'layanan']:
            tabel = audit.groupby(grup).agg(jumlah=('id_paket', 'size'), berat_kosong=('berat_kosong', 'sum'))
            tabel['persen_berat_kosong'] = 100 * tabel.berat_kosong / tabel.jumlah
            tabel.to_csv(args.output_dir / f'berat_kosong_{nama}_per_{grup}.csv')
    print(json.dumps(hasil, indent=2, ensure_ascii=False))
    print('\n10 korelasi Spearman terbesar (nilai absolut):')
    print(corr.head(10).to_string())
    print(f'\nRingkasan tersimpan di: {args.output_dir}')
    print('Tidak ada data yang dibuang, diimputasi, atau dipakai melatih model.')

In [ ]:
jalankan_audit_lanjutan_regresi()

## 3. Validasi waktu
Latih dari masa lalu dan nilai pada bulan berikutnya. Label latihan harus sudah tersedia saat batas waktu.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Menyiapkan indeks train/validasi secara kronologis, belum melatih model."""
from pathlib import Path
import numpy as np
import pandas as pd

def buat_fold(data, mulai, selesai):
    """Indeks posisi baris; label train dianggap tersedia setelah paket tiba.

    Asumsi waktu tiba = tanggal pickup + jam pickup + durasi.
    Validasi tetap memakai semua label yang diberikan, termasuk yang ekstrem.
    """
    pickup = pd.to_datetime(data['tgl_pickup'], errors='raise') + pd.to_timedelta(data['jam_pickup'], unit='h')
    durasi = pd.to_numeric(data['durasi_jam'], errors='raise')
    if pickup.isna().any() or not np.isfinite(durasi).all() or (durasi <= 0).any():
        raise ValueError('Periksa tanggal atau target tidak valid sebelum membuat fold.')
    tiba = pickup + pd.to_timedelta(durasi, unit='h')
    batas, akhir = (pd.Timestamp(mulai), pd.Timestamp(selesai))
    sebelum = pickup < batas
    train = sebelum & (tiba < batas)
    validasi = (pickup >= batas) & (pickup < akhir)
    train_idx, validasi_idx = (np.flatnonzero(train), np.flatnonzero(validasi))
    if not len(train_idx) or not len(validasi_idx):
        raise ValueError('Train atau validasi kosong.')
    assert not np.intersect1d(train_idx, validasi_idx).size
    assert pickup.iloc[train_idx].max() < pickup.iloc[validasi_idx].min()
    assert (tiba.iloc[train_idx] < batas).all()
    return (train_idx, validasi_idx, {'validasi_mulai': mulai, 'validasi_selesai_eksklusif': selesai, 'train_sebelum_cek_label': int(sebelum.sum()), 'train_label_belum_tersedia': int((sebelum & ~train).sum()), 'train_dipakai': len(train_idx), 'validasi': len(validasi_idx)})

def jalankan_validasi_waktu_regresi():
    repo = (REPO / 'kode/notebook.py').parents[1]
    args = SimpleNamespace(data_dir=DATA_DIR, output_dir=repo / 'kode/hasil/validasi_regresi')
    data = pd.read_csv(args.data_dir / 'regresi_train.csv.gz')
    hasil = []
    for bulan, berikutnya in [('06', '07'), ('07', '08'), ('08', '09')]:
        _, _, ringkasan = buat_fold(data, f'2025-{bulan}-01', f'2025-{berikutnya}-01')
        hasil.append(ringkasan)
    tabel = pd.DataFrame(hasil)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    tabel.to_csv(args.output_dir / 'ringkasan_fold.csv', index=False)
    print(tabel.to_string(index=False))
    print('\nLolos: train mendahului validasi; indeks tidak tumpang tindih; label train sudah tersedia menurut asumsi.')
    print('Semua baris periode validasi tetap dinilai. Dataset sumber tidak diubah.')
    print('Ini fold eksplorasi baseline; belum pembagian final tuning/kalibrasi/evaluasi.')

In [ ]:
jalankan_validasi_waktu_regresi()

## 4. Baseline
Buat patokan sederhana: kuantil global dan kelompok layanan–pulau.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Baseline kuantil global dan layanan x pulau, dievaluasi pada fold waktu."""
import json
import platform
from importlib.metadata import version
from pathlib import Path
import numpy as np
import pandas as pd
KOLOM = ['layanan', 'pulau_tujuan']

def kategori(data):
    return data[KOLOM].apply(lambda x: x.astype('string').str.strip().str.lower().fillna('__missing__'))

def latih_baseline(train):
    """Semua kuantil dipelajari dari train saja, tanpa jumlah_scan atau ID."""
    target = train['durasi_jam']
    global_q = target.quantile([0.5, 0.9]).to_numpy()
    tabel = kategori(train).assign(durasi_jam=target.to_numpy())
    kelompok = tabel.groupby(KOLOM)['durasi_jam'].quantile([0.5, 0.9]).unstack()
    kelompok.columns = ['p50', 'p90']
    return (global_q, kelompok)

def prediksi_baseline(model, data):
    global_q, kelompok = model
    keys = pd.MultiIndex.from_frame(kategori(data))
    pred = kelompok.reindex(keys).to_numpy(copy=True)
    fallback = np.isnan(pred).any(axis=1)
    pred[fallback] = global_q
    if not np.isfinite(pred).all() or (pred[:, 1] < pred[:, 0]).any():
        raise ValueError('Prediksi tidak valid atau p90 lebih kecil dari p50.')
    return (pred, int(fallback.sum()))

def metrik(y, pred):
    hasil = {}
    for i, q in enumerate([0.5, 0.9]):
        error = y - pred[:, i]
        hasil[f'pinball_p{int(q * 100)}'] = float(np.maximum(q * error, (q - 1) * error).mean())
        hasil[f'coverage_p{int(q * 100)}'] = float((y <= pred[:, i]).mean())
    hasil['mae_p50_jam'] = float(np.abs(y - pred[:, 0]).mean())
    return hasil

def jalankan_baseline_regresi():
    repo = (REPO / 'kode/notebook.py').parents[1]
    args = SimpleNamespace(data_dir=DATA_DIR, output_dir=repo / 'kode/hasil/baseline_regresi')
    data = pd.read_csv(args.data_dir / 'regresi_train.csv.gz')
    args.output_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for bulan, akhir in [('06', '07'), ('07', '08'), ('08', '09')]:
        tr, va, info = buat_fold(data, f'2025-{bulan}-01', f'2025-{akhir}-01')
        train, validasi = (data.iloc[tr], data.iloc[va])
        model = latih_baseline(train)
        pred, fallback = prediksi_baseline(model, validasi)
        global_pred = np.tile(model[0], (len(validasi), 1))
        for nama, nilai, n_fallback in [('global', global_pred, 0), ('layanan_pulau', pred, fallback)]:
            rows.append({'bulan': f'2025-{bulan}', 'model': nama, **info, 'fallback_global': n_fallback, **metrik(validasi.durasi_jam.to_numpy(), nilai)})
        model[1].to_csv(args.output_dir / f'kuantil_train_{bulan}.csv')
    hasil = pd.DataFrame(rows)
    hasil.to_csv(args.output_dir / 'metrik_per_fold.csv', index=False)
    nama_metrik = list(metrik(np.array([1.0]), np.array([[1.0, 1.0]])).keys())
    gabungan = []
    for nama, bagian in hasil.groupby('model'):
        gabungan.append({'model': nama, 'jumlah_validasi': int(bagian.validasi.sum()), **{m: float(np.average(bagian[m], weights=bagian.validasi)) for m in nama_metrik}})
    pd.DataFrame(gabungan).to_csv(args.output_dir / 'metrik_gabungan.csv', index=False)
    metadata = {'python': platform.python_version(), 'paket': {p: version(p) for p in ['numpy', 'pandas']}, 'acak': False, 'kuantil_interpolasi': 'linear', 'label_train': 'pickup + durasi < awal validasi', 'target_validasi': 'semua label diberikan, termasuk ekstrem', 'status': 'eksplorasi; belum kalibrasi atau prediksi test'}
    (args.output_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print(hasil[['bulan', 'model', 'train_dipakai', 'validasi', 'fallback_global', 'pinball_p50', 'pinball_p90', 'coverage_p90']].to_string(index=False))
    print('\nGabungan berbobot jumlah baris validasi:')
    print(pd.DataFrame(gabungan).to_string(index=False))
    print(f'\nHasil tersimpan di {args.output_dir}')
    print('Belum membuat prediksi test atau mengkalibrasi P90. Dataset mentah tidak diubah.')

In [ ]:
jalankan_baseline_regresi()

## 5. Perbandingan validasi
Bandingkan pembagian acak dan waktu. Periode dan ukuran train berbeda sehingga selisih bukan efek murni pengacakan.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Bandingkan baseline identik pada fold waktu dan KFold acak."""
import json
from importlib.metadata import version
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

def jalankan_bandingkan_validasi_regresi():
    repo = (REPO / 'kode/notebook.py').parents[1]
    args = SimpleNamespace(data_dir=DATA_DIR, output_dir=repo / 'kode/hasil/perbandingan_validasi')
    data = pd.read_csv(args.data_dir / 'regresi_train.csv.gz')
    folds = []
    for bulan, akhir in [('06', '07'), ('07', '08'), ('08', '09')]:
        tr, va, info = buat_fold(data, f'2025-{bulan}-01', f'2025-{akhir}-01')
        folds.append(('waktu', bulan, tr, va, info['train_label_belum_tersedia']))
    for i, (tr, va) in enumerate(KFold(n_splits=3, shuffle=True, random_state=42).split(data), 1):
        folds.append(('acak', str(i), tr, va, None))
    rows = []
    metrik_cols = None
    for skema, fold, tr, va, belum in folds:
        assert not np.intersect1d(tr, va).size
        train, validasi = (data.iloc[tr], data.iloc[va])
        pred, fallback = prediksi_baseline(latih_baseline(train), validasi)
        nilai = metrik(validasi.durasi_jam.to_numpy(), pred)
        metrik_cols = list(nilai)
        rows.append({'skema': skema, 'fold': fold, 'train': len(tr), 'validasi': len(va), 'train_label_belum_tersedia': belum, 'fallback_global': fallback, 'train_pickup_min': train.tgl_pickup.min(), 'train_pickup_max': train.tgl_pickup.max(), 'validasi_pickup_min': validasi.tgl_pickup.min(), 'validasi_pickup_max': validasi.tgl_pickup.max(), **nilai})
    hasil = pd.DataFrame(rows)
    gabungan = pd.DataFrame([{'skema': skema, 'jumlah_validasi': int(grup.validasi.sum()), **{m: float(np.average(grup[m], weights=grup.validasi)) for m in metrik_cols}} for skema, grup in hasil.groupby('skema')])
    args.output_dir.mkdir(parents=True, exist_ok=True)
    hasil.to_csv(args.output_dir / 'metrik_per_fold.csv', index=False)
    gabungan.to_csv(args.output_dir / 'metrik_gabungan.csv', index=False)
    catatan = {'seed': 42, 'model': 'kuantil layanan x pulau; fallback kuantil global train', 'paket': {p: version(p) for p in ['pandas', 'numpy', 'scikit-learn']}, 'waktu': 'Validasi Juni-Agustus; train sebelum cutoff dan label diperkirakan sudah tersedia.', 'acak': '3-fold KFold Januari-Agustus, tanpa cutoff atau penyaringan kematangan label.', 'batasan': 'Periode evaluasi dan ukuran train berbeda. Selisih skor bukan efek murni pengacakan. Fold acak dapat memakai masa depan untuk memprediksi masa lalu dan bukan simulasi deployment.', 'status': 'Eksplorasi baseline, belum model lanjutan atau evaluasi test.'}
    (args.output_dir / 'metadata.json').write_text(json.dumps(catatan, indent=2), encoding='utf-8')
    print(hasil[['skema', 'fold', 'train', 'validasi', 'pinball_p50', 'pinball_p90', 'coverage_p90']].to_string(index=False))
    print('\nGabungan berbobot jumlah baris:')
    print(gabungan.to_string(index=False))
    print('\nCatatan:', catatan['batasan'])
    print(f'\nHasil disimpan: {args.output_dir}')

In [ ]:
jalankan_bandingkan_validasi_regresi()

## 6. Model LightGBM
Latih P50 dan P90 pada pengembangan Mei–Juni; pilih kandidat dengan rata-rata pinball loss terkecil.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Pengembangan LightGBM kuantil pada Mei-Juni, tanpa kalibrasi/test."""
import json
import platform
from importlib.metadata import version
from pathlib import Path
import lightgbm as lgb
import numpy as np
import pandas as pd
KATEGORI_model_lightgbm_regresi = ['layanan', 'mitra_kurir', 'hub_asal', 'kota_asal', 'pulau_tujuan', 'kota_tujuan']
NUMERIK = ['jam_pickup', 'jarak_km', 'berat_kg', 'volume_liter', 'jumlah_item', 'nilai_barang', 'asuransi', 'cod', 'alamat_skor', 'pengalaman_kurir_bulan', 'rating_penjual', 'umur_akun_penjual_hari', 'volume_penjual_30h', 'antrean_hub_asal', 'indeks_kemacetan', 'hujan_asal_mm', 'hujan_tujuan_mm']
NUMERIK += [f'f{i:02d}' for i in range(1, 51)]
KANDIDAT = {'lgb_15': {'num_leaves': 15, 'min_child_samples': 100}, 'lgb_31': {'num_leaves': 31, 'min_child_samples': 100}}
PARAMETER = dict(n_estimators=200, learning_rate=0.05, reg_lambda=1.0, objective='quantile', random_state=42, n_jobs=4, deterministic=True, force_col_wise=True, verbosity=-1)

def fitur(data, kamus=None):
    """Kamus kategori dipelajari pada train; kategori baru menjadi missing."""
    x = data[NUMERIK].copy()
    tgl = pd.to_datetime(data.tgl_pickup, errors='raise')
    x['hari_minggu'] = tgl.dt.day_of_week
    x['bulan_pickup'] = tgl.dt.month
    x['berat_kosong'] = data.berat_kg.isna().astype('int8')
    belajar = kamus is None
    kamus = {} if belajar else kamus
    baru = {}
    for kolom in KATEGORI_model_lightgbm_regresi:
        nilai = data[kolom].astype('string').str.strip().str.lower()
        if belajar:
            kamus[kolom] = sorted(nilai.dropna().unique().tolist())
        baru[kolom] = int((nilai.notna() & ~nilai.isin(kamus[kolom])).sum())
        dikenal = nilai.where(nilai.isin(kamus[kolom]))
        x[kolom] = pd.Categorical(dikenal, categories=kamus[kolom])
    return (x, kamus, baru)

def jalankan_model_lightgbm_regresi():
    repo = (REPO / 'kode/notebook.py').parents[1]
    args = SimpleNamespace(data_dir=DATA_DIR, output_dir=repo / 'kode/hasil/lightgbm_regresi')
    data = pd.read_csv(args.data_dir / 'regresi_train.csv.gz')
    args.output_dir.mkdir(parents=True, exist_ok=True)
    rows, audit_kategori = ([], [])
    for bulan, akhir in [('05', '06'), ('06', '07')]:
        tr, va, info = buat_fold(data, f'2025-{bulan}-01', f'2025-{akhir}-01')
        train, validasi = (data.iloc[tr], data.iloc[va])
        x_train, kamus, _ = fitur(train)
        x_val, _, baru = fitur(validasi, kamus)
        audit_kategori.append({'bulan': bulan, **baru})
        y = validasi.durasi_jam.to_numpy()
        base, fallback = prediksi_baseline(latih_baseline(train), validasi)
        rows.append({'bulan': bulan, 'model': 'baseline', **info, 'crossing_sebelum_koreksi': 0, **metrik(y, base)})
        for nama, parameter in KANDIDAT.items():
            pred = []
            for q in [0.5, 0.9]:
                print(f'Validasi bulan {bulan}: melatih {nama}, P{int(100 * q)}...', flush=True)
                model = lgb.LGBMRegressor(alpha=q, **PARAMETER, **parameter)
                model.fit(x_train, train.durasi_jam, categorical_feature=KATEGORI_model_lightgbm_regresi)
                pred.append(model.predict(x_val))
            pred = np.column_stack(pred)
            crossing = int((pred[:, 1] < pred[:, 0]).sum())
            pred[:, 0] = np.maximum(pred[:, 0], 0)
            pred[:, 1] = np.maximum(pred[:, 1], pred[:, 0])
            if not np.isfinite(pred).all():
                raise ValueError('Prediksi mengandung nilai tidak finite.')
            rows.append({'bulan': bulan, 'model': nama, **info, 'crossing_sebelum_koreksi': crossing, **metrik(y, pred)})
    hasil = pd.DataFrame(rows)
    hasil.to_csv(args.output_dir / 'metrik_per_fold.csv', index=False)
    cols = list(metrik(np.array([1]), np.array([[1, 1]])).keys())
    gabungan = pd.DataFrame([{'model': nama, 'jumlah_validasi': int(g.validasi.sum()), **{m: float(np.average(g[m], weights=g.validasi)) for m in cols}} for nama, g in hasil.groupby('model')])
    gabungan['skor_pemilihan'] = (gabungan.pinball_p50 + gabungan.pinball_p90) / 2
    gabungan.to_csv(args.output_dir / 'metrik_gabungan.csv', index=False)
    kandidat = gabungan[gabungan.model != 'baseline'].sort_values('skor_pemilihan')
    terbaik = kandidat.iloc[0].model
    metadata = {'python': platform.python_version(), 'versi': {p: version(p) for p in ['lightgbm', 'numpy', 'pandas', 'scikit-learn']}, 'parameter_tetap': PARAMETER, 'kandidat': KANDIDAT, 'kandidat_terpilih': terbaik, 'parameter_terpilih': KANDIDAT[terbaik], 'kriteria': 'rata-rata pinball P50/P90, berbobot baris validasi Mei-Juni', 'fitur': x_train.columns.tolist(), 'dikecualikan': ['id_paket', 'jumlah_scan', 'durasi_jam'], 'catatan': 'Dua konfigurasi satu algoritme, bukan dua model lanjutan berbeda. Skor adalah skor pemilihan; belum evaluasi independen atau kalibrasi. Target validasi ekstrem dipertahankan. Train mengikuti asumsi waktu tiba.'}
    (args.output_dir / 'konfigurasi.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    pd.DataFrame(audit_kategori).to_csv(args.output_dir / 'kategori_baru_validasi.csv', index=False)
    print('\nHasil pengembangan Mei-Juni:')
    print(gabungan.to_string(index=False))
    print(f'\nKandidat LightGBM terpilih: {terbaik}')
    print('Belum mengevaluasi Juli/Agustus atau test; belum kalibrasi. Model akan dilatih ulang untuk tahap berikutnya.')

In [ ]:
jalankan_model_lightgbm_regresi()

## 7. Model linear Ridge
Bandingkan model linear; kuantil residual ditentukan dari bagian waktu terpisah dalam data latihan.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Ridge pada log1p durasi + kuantil residual dari periode train terpisah."""
import json
import platform
from importlib.metadata import version
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
ALPHAS = [1.0, 100.0]

def fitur_linear(data):
    x = data[NUMERIK].copy()
    tanggal = pd.to_datetime(data.tgl_pickup, errors='raise')
    x['hari_minggu'] = tanggal.dt.day_of_week
    x['bulan_pickup'] = tanggal.dt.month
    x['berat_kosong'] = data.berat_kg.isna().astype('int8')
    for kolom in KATEGORI_model_lightgbm_regresi:
        x[kolom] = data[kolom].astype('string').str.strip().str.lower().fillna('__missing__').astype(object)
    return x

def pipeline_linear(alpha):
    numerik = NUMERIK + ['hari_minggu', 'bulan_pickup', 'berat_kosong']
    preproses = ColumnTransformer([('numerik', Pipeline([('isi_kosong', SimpleImputer(strategy='median', keep_empty_features=True)), ('skala', StandardScaler())]), numerik), ('kategori', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=50, sparse_output=True), KATEGORI_model_lightgbm_regresi)], sparse_threshold=1.0)
    return Pipeline([('preproses', preproses), ('ridge', Ridge(alpha=alpha, solver='lsqr', tol=1e-06, max_iter=2000))])

def prediksi_kuantil(model, x, residual_q):
    pred_log = model.predict(x)[:, None] + residual_q[None, :]
    with np.errstate(over='raise', invalid='raise'):
        pred = np.maximum(np.expm1(pred_log), 0)
    if not np.isfinite(pred).all() or (pred[:, 1] < pred[:, 0]).any():
        raise ValueError('Prediksi kuantil tidak valid.')
    return pred

def bagi_internal(data, awal_validasi, akhir_validasi):
    """Fit pada periode lebih awal; residual bulan terakhir sudah matang saat cutoff luar."""
    luar_tr, luar_va, info = buat_fold(data, awal_validasi, akhir_validasi)
    awal_residual = (pd.Timestamp(awal_validasi) - pd.DateOffset(months=1)).strftime('%Y-%m-%d')
    fit_idx, residual_semua, _ = buat_fold(data, awal_residual, awal_validasi)
    residual_idx = np.intersect1d(residual_semua, luar_tr)
    if len(residual_idx) == 0:
        raise ValueError('Periode residual kosong.')
    assert not np.intersect1d(fit_idx, residual_idx).size
    assert not np.intersect1d(fit_idx, luar_va).size
    assert not np.intersect1d(residual_idx, luar_va).size
    assert np.isin(fit_idx, luar_tr).all()
    return (fit_idx, residual_idx, luar_tr, luar_va, {**info, 'awal_residual': awal_residual, 'fit_ridge': len(fit_idx), 'residual_dipakai': len(residual_idx), 'residual_belum_matang': len(residual_semua) - len(residual_idx)})

def jalankan_model_linear_regresi():
    repo = (REPO / 'kode/notebook.py').parents[1]
    args = SimpleNamespace(data_dir=DATA_DIR, output_dir=repo / 'kode/hasil/linear_regresi')
    data = pd.read_csv(args.data_dir / 'regresi_train.csv.gz')
    args.output_dir.mkdir(parents=True, exist_ok=True)
    rows, residual_rows = ([], [])
    for bulan, akhir in [('05', '06'), ('06', '07')]:
        fit_idx, res_idx, tr, va, info = bagi_internal(data, f'2025-{bulan}-01', f'2025-{akhir}-01')
        fit, residual, validasi = (data.iloc[fit_idx], data.iloc[res_idx], data.iloc[va])
        x_fit, x_res, x_val = (fitur_linear(d) for d in [fit, residual, validasi])
        y = validasi.durasi_jam.to_numpy()
        base, _ = prediksi_baseline(latih_baseline(data.iloc[tr]), validasi)
        rows.append({'bulan': bulan, 'model': 'baseline', **info, **metrik(y, base)})
        for alpha in ALPHAS:
            nama = f'ridge_{alpha:g}'
            print(f'Validasi {bulan}: {nama}; fit {len(fit):,}, residual {len(residual):,} baris...', flush=True)
            model = pipeline_linear(alpha)
            model.fit(x_fit, np.log1p(fit.durasi_jam))
            errors = np.log1p(residual.durasi_jam.to_numpy()) - model.predict(x_res)
            residual_q = np.quantile(errors, [0.5, 0.9])
            pred = prediksi_kuantil(model, x_val, residual_q)
            rows.append({'bulan': bulan, 'model': nama, **info, **metrik(y, pred)})
            residual_rows.append({'bulan': bulan, 'model': nama, 'residual_log_p50': residual_q[0], 'residual_log_p90': residual_q[1], 'iterasi_solver': int(np.max(model.named_steps['ridge'].n_iter_))})
    hasil = pd.DataFrame(rows)
    hasil.to_csv(args.output_dir / 'metrik_per_fold.csv', index=False)
    pd.DataFrame(residual_rows).to_csv(args.output_dir / 'kuantil_residual.csv', index=False)
    cols = list(metrik(np.array([1]), np.array([[1, 1]])).keys())
    gabungan = pd.DataFrame([{'model': nama, 'jumlah_validasi': int(g.validasi.sum()), **{m: float(np.average(g[m], weights=g.validasi)) for m in cols}} for nama, g in hasil.groupby('model')])
    gabungan['skor_pemilihan'] = (gabungan.pinball_p50 + gabungan.pinball_p90) / 2
    gabungan.to_csv(args.output_dir / 'metrik_gabungan.csv', index=False)
    terbaik = gabungan[gabungan.model != 'baseline'].sort_values('skor_pemilihan').iloc[0].model
    metadata = {'python': platform.python_version(), 'versi': {p: version(p) for p in ['numpy', 'pandas', 'scipy', 'scikit-learn']}, 'alpha_kandidat': ALPHAS, 'kandidat_terpilih': terbaik, 'target': 'log1p(durasi_jam)', 'kuantil': 'expm1(prediksi_log + kuantil_residual_log)', 'residual': 'Bulan terakhir sebelum validasi, hanya label tersedia pada cutoff luar.', 'fit': 'Periode sebelum bulan residual; hanya label tersedia pada cutoff fit.', 'fitur': x_fit.columns.tolist(), 'min_frequency': 50, 'solver': 'lsqr', 'tol': 1e-06, 'max_iter': 2000, 'acak': False, 'kriteria': 'Rata-rata pinball P50/P90 berbobot jumlah baris validasi Mei-Juni.', 'batasan': 'Kuantil residual global mengasumsikan pola residual log relatif stabil. Bukan jaminan coverage kondisional atau conformal. Model dasar tidak di-refit setelah residual. Ukuran fit Ridge lebih kecil daripada LightGBM karena pemisahan residual. Skor pemilihan bukan evaluasi independen; Juli/Agustus belum dievaluasi oleh skrip ini.'}
    (args.output_dir / 'konfigurasi.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print('\nPengembangan model linear Mei-Juni:')
    print(gabungan.to_string(index=False))
    print(f'\nKandidat Ridge terpilih: {terbaik}')
    print('Belum kalibrasi Juli, evaluasi Agustus, atau prediksi test.')

In [ ]:
jalankan_model_linear_regresi()

## 8. Audit leakage
Uji jumlah_scan sebagai eksperimen. Model utama tetap mengecualikannya karena ketersediaan saat pickup belum pasti.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Eksperimen terkontrol jumlah_scan, bukan pembuktian waktu tersedianya fitur."""
import json
import platform
from importlib.metadata import version
from pathlib import Path
import lightgbm as lgb
import numpy as np
import pandas as pd

def pasangan_fitur(train, validasi):
    """Pastikan dua kondisi berbeda tepat satu kolom, tanpa mengubah fitur utama."""
    x_train, kamus, _ = fitur(train)
    x_val, _, _ = fitur(validasi, kamus)
    assert 'jumlah_scan' not in x_train and 'jumlah_scan' not in x_val
    train_scan, val_scan = (x_train.copy(), x_val.copy())
    train_scan['jumlah_scan'] = pd.to_numeric(train.jumlah_scan, errors='raise')
    val_scan['jumlah_scan'] = pd.to_numeric(validasi.jumlah_scan, errors='raise')
    pd.testing.assert_frame_equal(train_scan.drop(columns='jumlah_scan'), x_train)
    pd.testing.assert_frame_equal(val_scan.drop(columns='jumlah_scan'), x_val)
    return {'tanpa_scan': (x_train, x_val), 'dengan_scan': (train_scan, val_scan)}

def jalankan_audit_leakage_regresi():
    repo = (REPO / 'kode/notebook.py').parents[1]
    args = SimpleNamespace(data_dir=DATA_DIR, output_dir=repo / 'kode/hasil/audit_leakage_regresi')
    data = pd.read_csv(args.data_dir / 'regresi_train.csv.gz')
    args.output_dir.mkdir(parents=True, exist_ok=True)
    rows, importance = ([], [])
    for bulan, akhir in [('05', '06'), ('06', '07')]:
        tr, va, info = buat_fold(data, f'2025-{bulan}-01', f'2025-{akhir}-01')
        train, validasi = (data.iloc[tr], data.iloc[va])
        for kondisi, (x_train, x_val) in pasangan_fitur(train, validasi).items():
            pred = []
            for q in [0.5, 0.9]:
                print(f'Validasi {bulan}: {kondisi}, P{int(q * 100)}...', flush=True)
                model = lgb.LGBMRegressor(alpha=q, **PARAMETER, **KANDIDAT['lgb_31'])
                model.fit(x_train, train.durasi_jam, categorical_feature=KATEGORI_model_lightgbm_regresi)
                pred.append(model.predict(x_val))
                gain = model.booster_.feature_importance(importance_type='gain')
                for nama, nilai in zip(x_train.columns, gain):
                    importance.append({'bulan': bulan, 'kondisi': kondisi, 'kuantil': q, 'fitur': nama, 'gain': float(nilai), 'porsi_gain': float(nilai / gain.sum()) if gain.sum() else 0.0})
            pred = np.column_stack(pred)
            crossing = int((pred[:, 1] < pred[:, 0]).sum())
            pred[:, 0] = np.maximum(pred[:, 0], 0)
            pred[:, 1] = np.maximum(pred[:, 1], pred[:, 0])
            if not np.isfinite(pred).all():
                raise ValueError('Prediksi tidak finite.')
            rows.append({'bulan': bulan, 'kondisi': kondisi, **info, 'crossing_sebelum_koreksi': crossing, **metrik(validasi.durasi_jam.to_numpy(), pred)})
    hasil = pd.DataFrame(rows)
    hasil.to_csv(args.output_dir / 'metrik_per_fold.csv', index=False)
    pd.DataFrame(importance).to_csv(args.output_dir / 'importance_gain.csv', index=False)
    cols = list(metrik(np.array([1]), np.array([[1, 1]])).keys())
    gabungan = pd.DataFrame([{'kondisi': nama, 'jumlah_validasi': int(g.validasi.sum()), **{m: float(np.average(g[m], weights=g.validasi)) for m in cols}} for nama, g in hasil.groupby('kondisi')])
    gabungan.to_csv(args.output_dir / 'metrik_gabungan.csv', index=False)
    indeks = gabungan.set_index('kondisi')
    dampak = []
    for m in ['pinball_p50', 'pinball_p90']:
        sebelum, sesudah = (indeks.loc['tanpa_scan', m], indeks.loc['dengan_scan', m])
        dampak.append({'metrik': m, 'tanpa_scan': sebelum, 'dengan_scan': sesudah, 'penurunan_loss': sebelum - sesudah, 'penurunan_persen': 100 * (sebelum - sesudah) / sebelum})
    pd.DataFrame(dampak).to_csv(args.output_dir / 'dampak_scan.csv', index=False)
    metadata = {'python': platform.python_version(), 'versi': {p: version(p) for p in ['lightgbm', 'numpy', 'pandas', 'scikit-learn']}, 'parameter': {**PARAMETER, **KANDIDAT['lgb_31']}, 'desain': 'Mei-Juni; baris train/validasi, seed, preprocessing dan parameter sama; hanya tambah jumlah_scan.', 'tuning_ulang': False, 'ketersediaan_scan': 'Belum dikonfirmasi. Kamus hanya menyebut jumlah scan yang tercatat di sistem.', 'keputusan': 'Model utama tetap tanpa jumlah_scan sampai ketersediaan saat pickup dapat dipastikan.', 'batasan': 'Perbaikan skor atau gain tinggi bukan bukti waktu tersedianya fitur. Jika scan mencakup kejadian setelah pickup, penggunaan tersebut adalah leakage. Jika hanya scan yang sudah ada saat pickup, fitur berpotensi sah. Gain adalah statistik model, bukan hubungan sebab-akibat atau pengganti SHAP/PDP. Ini eksperimen pengembangan; belum kalibrasi/evaluasi akhir atau prediksi test.'}
    (args.output_dir / 'konfigurasi.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print('\nHasil gabungan Mei-Juni:')
    print(gabungan.to_string(index=False))
    print('\nDampak penambahan jumlah_scan (nilai positif = loss menurun):')
    print(pd.DataFrame(dampak).to_string(index=False))
    print('\nKeputusan:', metadata['keputusan'])
    print('Eksperimen tidak memastikan kapan scan dicatat. Dataset dan model utama tidak diubah.')

In [ ]:
jalankan_audit_leakage_regresi()

## 9. Uji ketahanan
Periksa efek pembatasan target ekstrem dan kurir yang tidak terlihat saat latihan.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Sensitivitas target ekstrem dan kurir baru pada pengembangan Mei-Juni."""
import json
import platform
from importlib.metadata import version
from pathlib import Path
import lightgbm as lgb
import numpy as np
import pandas as pd
SEED_uji_ketahanan_regresi = 42
BATAS_KUANTIL = 0.99

def normalisasi_uji_ketahanan_regresi(series):
    return series.astype('string').str.strip().str.lower()

def pilih_kurir(train):
    """Pemilihan tanpa melihat label atau kategori validasi/test."""
    tersedia = sorted(normalisasi_uji_ketahanan_regresi(train.mitra_kurir).dropna().unique())
    if len(tersedia) < 3:
        raise ValueError('Diperlukan minimal tiga kurir untuk menyembunyikan dua.')
    return sorted(np.random.default_rng(SEED_uji_ketahanan_regresi).choice(tersedia, size=2, replace=False).tolist())

def target_dibatasi(train):
    batas = float(train.durasi_jam.quantile(BATAS_KUANTIL))
    y = train.durasi_jam.clip(upper=batas)
    return (y, batas, int((train.durasi_jam > batas).sum()))

def latih_prediksi(train, validasi, target, label):
    x_train, kamus, _ = fitur(train)
    x_val, _, kategori_baru = fitur(validasi, kamus)
    pred = []
    for q in [0.5, 0.9]:
        print(f'{label}, P{int(100 * q)}; train {len(train):,} baris...', flush=True)
        model = lgb.LGBMRegressor(alpha=q, **PARAMETER, **KANDIDAT['lgb_31'])
        model.fit(x_train, target, categorical_feature=KATEGORI_model_lightgbm_regresi)
        pred.append(model.predict(x_val))
    pred = np.column_stack(pred)
    crossing = int((pred[:, 1] < pred[:, 0]).sum())
    pred[:, 0] = np.maximum(pred[:, 0], 0)
    pred[:, 1] = np.maximum(pred[:, 1], pred[:, 0])
    if not np.isfinite(pred).all():
        raise ValueError('Prediksi tidak finite.')
    return (pred, crossing, kategori_baru)

def jalankan_uji_ketahanan_regresi():
    repo = (REPO / 'kode/notebook.py').parents[1]
    args = SimpleNamespace(data_dir=DATA_DIR, output_dir=repo / 'kode/hasil/uji_ketahanan_regresi')
    data = pd.read_csv(args.data_dir / 'regresi_train.csv.gz')
    args.output_dir.mkdir(parents=True, exist_ok=True)
    rows, audit = ([], [])
    for bulan, akhir in [('05', '06'), ('06', '07')]:
        tr, va, _ = buat_fold(data, f'2025-{bulan}-01', f'2025-{akhir}-01')
        train, validasi = (data.iloc[tr], data.iloc[va])
        y_val = validasi.durasi_jam.to_numpy(copy=True)
        y_cap, batas, n_cap = target_dibatasi(train)
        kurir = pilih_kurir(train)
        simpan_train = ~normalisasi_uji_ketahanan_regresi(train.mitra_kurir).isin(kurir)
        train_baru = train.loc[simpan_train]
        target_kurir = normalisasi_uji_ketahanan_regresi(validasi.mitra_kurir).isin(kurir).to_numpy(dtype=bool)
        assert not normalisasi_uji_ketahanan_regresi(train_baru.mitra_kurir).isin(kurir).any()
        if not target_kurir.any():
            raise ValueError('Kurir yang disembunyikan tidak muncul pada validasi.')
        kondisi = [('asli', train, train.durasi_jam), ('cap_p99', train, y_cap), ('kurir_baru', train_baru, train_baru.durasi_jam)]
        for nama, latihan, y_train in kondisi:
            pred, crossing, baru = latih_prediksi(latihan, validasi, y_train, f'Validasi {bulan}: {nama}')
            if nama == 'kurir_baru':
                assert baru['mitra_kurir'] >= int(target_kurir.sum())
            subset = {'semua': np.ones(len(validasi), dtype=bool), 'kurir_disembunyikan': target_kurir, 'kurir_lain': ~target_kurir, 'target_di_atas_cap_train': y_val > batas}
            for nama_subset, mask in subset.items():
                if mask.any():
                    rows.append({'bulan': bulan, 'kondisi': nama, 'subset': nama_subset, 'jumlah_validasi': int(mask.sum()), 'train': len(latihan), **metrik(y_val[mask], pred[mask])})
            audit.append({'bulan': bulan, 'kondisi': nama, 'train': len(latihan), 'batas_p99_train_jam': batas, 'target_train_dibatasi': n_cap if nama == 'cap_p99' else 0, 'kurir_disembunyikan': '|'.join(kurir), 'baris_train_dikeluarkan': len(train) - len(latihan), 'kurir_baru_validasi': baru['mitra_kurir'], 'crossing_sebelum_koreksi': crossing})
            np.testing.assert_array_equal(validasi.durasi_jam.to_numpy(), y_val)
    hasil = pd.DataFrame(rows)
    hasil.to_csv(args.output_dir / 'metrik_per_fold.csv', index=False)
    pd.DataFrame(audit).to_csv(args.output_dir / 'audit_perlakuan.csv', index=False)
    cols = list(metrik(np.array([1]), np.array([[1, 1]])).keys())
    gabungan = pd.DataFrame([{'kondisi': kondisi, 'subset': subset, 'jumlah_validasi': int(g.jumlah_validasi.sum()), **{m: float(np.average(g[m], weights=g.jumlah_validasi)) for m in cols}} for (kondisi, subset), g in hasil.groupby(['kondisi', 'subset'])])
    gabungan['skor_pemilihan'] = (gabungan.pinball_p50 + gabungan.pinball_p90) / 2
    gabungan.to_csv(args.output_dir / 'metrik_gabungan.csv', index=False)
    dampak = []
    for kondisi, subset in [('cap_p99', 'semua'), ('kurir_baru', 'kurir_disembunyikan')]:
        a = gabungan[(gabungan.kondisi == 'asli') & (gabungan.subset == subset)].iloc[0]
        b = gabungan[(gabungan.kondisi == kondisi) & (gabungan.subset == subset)].iloc[0]
        assert a.jumlah_validasi == b.jumlah_validasi
        for m in ['pinball_p50', 'pinball_p90', 'coverage_p90']:
            dampak.append({'kondisi': kondisi, 'subset': subset, 'metrik': m, 'asli': a[m], 'perlakuan': b[m], 'selisih_perlakuan_minus_asli': b[m] - a[m]})
    pd.DataFrame(dampak).to_csv(args.output_dir / 'dampak.csv', index=False)
    meta = {'python': platform.python_version(), 'versi': {p: version(p) for p in ['numpy', 'pandas', 'lightgbm', 'scikit-learn']}, 'parameter': {**PARAMETER, **KANDIDAT['lgb_31']}, 'cap_kuantil': BATAS_KUANTIL, 'seed_kurir': SEED_uji_ketahanan_regresi, 'jumlah_kurir_disembunyikan': 2, 'desain': 'Mei-Juni. Semua kondisi tanpa jumlah_scan. Cap hanya target train. Validasi identik dan utuh.', 'keputusan': 'Eksperimen sensitivitas; tidak otomatis mengubah pipeline utama atau dataset.', 'batasan': 'Penghilangan kurir juga mengurangi jumlah dan mengubah komposisi train. Dampak bukan efek murni unknown encoding. Dua kurir dengan satu seed tidak mewakili semua kategori baru. Cap persentil bukan bukti target besar salah; label train sudah disaring menurut asumsi waktu tiba. Skor pengembangan bukan evaluasi akhir; Juli/Agustus/test belum dievaluasi.'}
    (args.output_dir / 'konfigurasi.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
    tampilan = gabungan[gabungan.subset.isin(['semua', 'kurir_disembunyikan'])]
    print('\nHasil pengembangan Mei-Juni:')
    print(tampilan[['kondisi', 'subset', 'jumlah_validasi', 'pinball_p50', 'pinball_p90', 'coverage_p90']].to_string(index=False))
    print('\nBaca perbandingan pada subset yang sama. Data validasi dan dataset mentah tidak diubah.')
    print('Model utama belum diubah. Hasil tersimpan di', args.output_dir)

In [ ]:
jalankan_uji_ketahanan_regresi()

## 10. Kalibrasi dan evaluasi
Latih sebelum Juli, koreksi P90 dari label Juli yang tersedia, lalu evaluasi Agustus. Agustus pernah dieksplorasi sehingga bukan holdout yang sepenuhnya belum dilihat.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Kalibrasi batas atas P90 pada Juli, evaluasi historis terpisah pada Agustus."""
import json
import math
import platform
from importlib.metadata import version
from pathlib import Path
import lightgbm as lgb
import numpy as np
import pandas as pd

def bagi_kalibrasi(data):
    train_idx, juli_idx, info_train = buat_fold(data, '2025-07-01', '2025-08-01')
    tersedia_agustus, evaluasi_idx, _ = buat_fold(data, '2025-08-01', '2025-09-01')
    kalibrasi_idx = np.intersect1d(juli_idx, tersedia_agustus)
    if not len(kalibrasi_idx):
        raise ValueError('Data kalibrasi kosong.')
    for a, b in [(train_idx, kalibrasi_idx), (train_idx, evaluasi_idx), (kalibrasi_idx, evaluasi_idx)]:
        assert not np.intersect1d(a, b).size
    return (train_idx, kalibrasi_idx, evaluasi_idx, {'train': len(train_idx), 'train_label_belum_tersedia': info_train['train_label_belum_tersedia'], 'juli_total': len(juli_idx), 'kalibrasi_juli': len(kalibrasi_idx), 'juli_belum_tiba_sebelum_agustus': len(juli_idx) - len(kalibrasi_idx), 'evaluasi_agustus': len(evaluasi_idx)})

def hitung_koreksi(y_kalibrasi, p90_kalibrasi, q=0.9):
    """Skor signed y-P90; statistik urut ceil((n+1)*q), bukan interpolasi."""
    y = np.asarray(y_kalibrasi, dtype=float)
    p90 = np.asarray(p90_kalibrasi, dtype=float)
    if y.ndim != 1 or y.shape != p90.shape or (not 0 < q < 1):
        raise ValueError('Bentuk input atau kuantil tidak valid.')
    skor = y - p90
    if not np.isfinite(skor).all():
        raise ValueError('Skor kalibrasi harus finite.')
    k = math.ceil((len(skor) + 1) * q)
    if not len(skor) or k > len(skor):
        raise ValueError('Sampel kalibrasi terlalu sedikit untuk koreksi finite pada kuantil ini.')
    return (float(np.partition(skor, k - 1)[k - 1]), k)

def terapkan_koreksi(pred, koreksi):
    pred = np.asarray(pred, dtype=float)
    if pred.ndim != 2 or pred.shape[1] != 2 or (not np.isfinite(pred).all()) or (not np.isfinite(koreksi)):
        raise ValueError('Prediksi atau koreksi tidak valid.')
    hasil = pred.copy()
    hasil[:, 1] = np.maximum(pred[:, 1] + koreksi, pred[:, 0])
    return hasil

def latih_pasangan(train):
    x_train, kamus, _ = fitur(train)
    models = []
    for q in [0.5, 0.9]:
        print(f'Melatih LightGBM P{int(q * 100)} pada {len(train):,} baris latihan...', flush=True)
        model = lgb.LGBMRegressor(alpha=q, **PARAMETER, **KANDIDAT['lgb_31'])
        model.fit(x_train, train.durasi_jam, categorical_feature=KATEGORI_model_lightgbm_regresi)
        models.append(model)
    return (models, kamus)

def prediksi_pasangan(models, kamus, data):
    x, _, baru = fitur(data, kamus)
    pred = np.column_stack([m.predict(x) for m in models])
    crossing = int((pred[:, 1] < pred[:, 0]).sum())
    pred[:, 0] = np.maximum(pred[:, 0], 0)
    pred[:, 1] = np.maximum(pred[:, 1], pred[:, 0])
    if not np.isfinite(pred).all():
        raise ValueError('Prediksi tidak finite.')
    return (pred, crossing, baru)

def jalankan_kalibrasi_regresi():
    repo = (REPO / 'kode/notebook.py').parents[1]
    args = SimpleNamespace(data_dir=DATA_DIR, output_dir=repo / 'kode/hasil/kalibrasi_regresi')
    data = pd.read_csv(args.data_dir / 'regresi_train.csv.gz')
    tr, ca, ev, pembagian = bagi_kalibrasi(data)
    train, kalibrasi, evaluasi = (data.iloc[tr], data.iloc[ca], data.iloc[ev])
    models, kamus = latih_pasangan(train)
    pred_ca, crossing_ca, baru_ca = prediksi_pasangan(models, kamus, kalibrasi)
    delta, urutan = hitung_koreksi(kalibrasi.durasi_jam.to_numpy(), pred_ca[:, 1])
    print(f'Koreksi P90 dari Juli: {delta:+.6f} jam (statistik urut {urutan}).', flush=True)
    pred_ev, crossing_ev, baru_ev = prediksi_pasangan(models, kamus, evaluasi)
    rows = []
    base = latih_baseline(train)
    for periode, frame, pred in [('juli_kalibrasi_diagnostik', kalibrasi, pred_ca), ('agustus_evaluasi', evaluasi, pred_ev)]:
        pred_base, _ = prediksi_baseline(base, frame)
        sesudah = terapkan_koreksi(pred, delta)
        for nama, p in [('baseline_tanpa_kalibrasi', pred_base), ('lgb_sebelum', pred), ('lgb_sesudah', sesudah)]:
            rows.append({'periode': periode, 'model': nama, 'jumlah': len(frame), **metrik(frame.durasi_jam.to_numpy(), p)})
        np.testing.assert_array_equal(pred[:, 0], sesudah[:, 0])
        assert (sesudah[:, 1] >= sesudah[:, 0]).all()
    hasil = pd.DataFrame(rows)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    hasil.to_csv(args.output_dir / 'metrik.csv', index=False)
    pd.DataFrame([pembagian]).to_csv(args.output_dir / 'pembagian_data.csv', index=False)
    pd.DataFrame([{'periode': 'juli', 'crossing': crossing_ca, **baru_ca}, {'periode': 'agustus', 'crossing': crossing_ev, **baru_ev}]).to_csv(args.output_dir / 'audit_prediksi.csv', index=False)
    meta = {'python': platform.python_version(), 'versi': {p: version(p) for p in ['numpy', 'pandas', 'lightgbm', 'scikit-learn']}, 'parameter': {**PARAMETER, **KANDIDAT['lgb_31']}, 'koreksi_p90_jam': delta, 'urutan_koreksi': urutan, 'q': 0.9, 'aturan': 'Skor y-P90 pada Juli matang. k=ceil((n+1)*0.9). P90=max(P90+skor_urut_k,P50).', 'fit': 'Sebelum Juli, hanya label diperkirakan tersedia pada 1 Juli.', 'kalibrasi': 'Pickup Juli dan tiba sebelum 1 Agustus.', 'evaluasi': 'Seluruh pickup Agustus dengan label yang diberikan dosen; tidak menyaring berdasarkan waktu tiba.', 'p50': 'Tidak dikalibrasi; dilaporkan apa adanya.', 'refit_setelah_kalibrasi': False, 'batasan': 'Koreksi bergaya split conformal satu sisi. Pergeseran waktu dan seleksi label yang sudah tiba membuat asumsi exchangeability tidak terjamin; tidak menjamin coverage 90%. Agustus telah dilihat saat audit/baseline; evaluasi historis, bukan holdout belum pernah dilihat. Pemilihan hyperparameter sebelumnya menggunakan label Mei-Juni yang diberikan, termasuk label yang mungkin belum tersedia pada cutoff Juli; bukan backtest operasional sepenuhnya. Skor Juli diagnostik saja. Jangan memilih ulang koreksi berdasarkan Agustus. Koreksi ini milik model yang dilatih sampai sebelum Juli; tidak otomatis sah untuk model refit baru.', 'referensi': 'https://arxiv.org/abs/2107.07511'}
    (args.output_dir / 'konfigurasi.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
    print('\nPembagian data:')
    print(pd.DataFrame([pembagian]).to_string(index=False))
    print('\nEvaluasi Agustus (model tetap, seluruh label Agustus):')
    print(hasil[hasil.periode == 'agustus_evaluasi'].drop(columns='periode').to_string(index=False))
    print('\nMetrik Juli tersimpan sebagai diagnostik, bukan evaluasi independen.')
    print('Belum membuat prediksi test atau melakukan refit model final. Hasil:', args.output_dir)

In [ ]:
jalankan_kalibrasi_regresi()

## 11. Analisis kesalahan
Cari layanan, pulau, atau kurir dengan kesalahan besar. Coverage global tidak menjamin coverage setiap kelompok.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Analisis kelompok pada Agustus; tidak memilih ulang model atau koreksi."""
import json
import platform
from importlib.metadata import version
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
DIMENSI = ['layanan', 'pulau_tujuan', 'mitra_kurir']

def tabel_kelompok(data, pred, kolom, nama_model):
    """Metrik tiap kelompok; bobot jumlah baris harus merekonstruksi metrik total."""
    if len(data) != len(pred) or len(data) == 0:
        raise ValueError('Ukuran data dan prediksi harus sama dan tidak kosong.')
    y = data.durasi_jam.to_numpy(dtype=float)
    pred = np.asarray(pred, dtype=float)
    if pred.shape != (len(data), 2) or not np.isfinite(pred).all() or (not np.isfinite(y).all()):
        raise ValueError('Target/prediksi tidak valid.')
    e90 = y - pred[:, 1]
    loss90 = np.maximum(0.9 * e90, -0.1 * e90)
    total_loss = loss90.sum()
    grup = data[kolom].astype('string').str.strip().fillna('(kosong)').to_numpy()
    rows = []
    for nilai in sorted(set(grup)):
        mask = grup == nilai
        rows.append({'model': nama_model, 'kelompok': nilai, 'jumlah': int(mask.sum()), 'porsi_data_persen': 100 * mask.mean(), **metrik(y[mask], pred[mask]), 'median_durasi_jam': float(np.median(y[mask])), 'durasi_di_atas_720_jam': int((y[mask] > 720).sum()), 'melewati_p90': int((y[mask] > pred[mask, 1]).sum()), 'kontribusi_loss_p90_persen': 100 * loss90[mask].sum() / total_loss if total_loss else 0.0})
    tabel = pd.DataFrame(rows)
    assert tabel.jumlah.sum() == len(data)
    for m, nilai in metrik(y, pred).items():
        np.testing.assert_allclose(np.average(tabel[m], weights=tabel.jumlah), nilai, rtol=1e-10, atol=1e-12)
    return tabel

def diagnostik_ekstrem(data, pred):
    y = data.durasi_jam.to_numpy()
    error = np.abs(y - pred[:, 0])
    e90 = y - pred[:, 1]
    loss90 = np.maximum(0.9 * e90, -0.1 * e90)
    rows = []
    for nama, mask in [('durasi_sampai_720_jam', y <= 720), ('durasi_di_atas_720_jam', y > 720)]:
        if mask.any():
            rows.append({'subset': nama, 'jumlah': int(mask.sum()), 'porsi_data_persen': 100 * mask.mean(), 'kontribusi_error_absolut_p50_persen': 100 * error[mask].sum() / error.sum() if error.sum() else 0.0, 'kontribusi_loss_p90_persen': 100 * loss90[mask].sum() / loss90.sum() if loss90.sum() else 0.0, **metrik(y[mask], pred[mask])})
    return pd.DataFrame(rows)

def gambar_coverage(tabel, path):
    fig, axes = plt.subplots(1, 3, figsize=(15, 6), layout='constrained')
    colors = {'baseline_tanpa_kalibrasi': '#94a3b8', 'lgb_sebelum': '#d97706', 'lgb_sesudah': '#087f8c'}
    labels = ['Baseline', 'LightGBM sebelum', 'LightGBM sesudah']
    for ax, dimensi in zip(axes, DIMENSI):
        t = tabel[dimensi]
        urutan = t[t.model == 'lgb_sesudah'].sort_values('coverage_p90').kelompok.tolist()
        y = np.arange(len(urutan))
        for i, (model, color) in enumerate(colors.items()):
            values = t[t.model == model].set_index('kelompok').loc[urutan]
            ax.barh(y + (i - 1) * 0.23, values.coverage_p90 * 100, height=0.21, color=color, label=labels[i])
        counts = t[t.model == 'lgb_sesudah'].set_index('kelompok').loc[urutan].jumlah
        ax.set_yticks(y, [f'{k} (n={n:,})' for k, n in zip(urutan, counts)])
        ax.tick_params(axis='y', labelsize=8)
        ax.axvline(90, color='#334155', linestyle='--', linewidth=1)
        ax.set_xlim(0, 100)
        ax.set_xlabel('Coverage P90 (%)')
        ax.set_title(dimensi.replace('_', ' ').title())
        ax.grid(axis='x', alpha=0.15)
        ax.set_axisbelow(True)
    fig.legend(*axes[0].get_legend_handles_labels(), loc='outside lower center', ncol=3, fontsize=9)
    fig.suptitle('Evaluasi Agustus: cakupan P90 per kelompok\nGaris putus-putus = sasaran 90%; model dan koreksi tetap', fontsize=14)
    fig.savefig(path, dpi=160)
    plt.close(fig)

def jalankan_analisis_error_regresi():
    repo = (REPO / 'kode/notebook.py').parents[1]
    args = SimpleNamespace(data_dir=DATA_DIR, output_dir=repo / 'kode/hasil/analisis_error_regresi')
    data = pd.read_csv(args.data_dir / 'regresi_train.csv.gz')
    tr, ca, ev, pembagian = bagi_kalibrasi(data)
    train, kalibrasi, evaluasi = (data.iloc[tr], data.iloc[ca], data.iloc[ev])
    models, kamus = latih_pasangan(train)
    p_cal, _, _ = prediksi_pasangan(models, kamus, kalibrasi)
    delta, _ = hitung_koreksi(kalibrasi.durasi_jam.to_numpy(), p_cal[:, 1])
    pred, _, _ = prediksi_pasangan(models, kamus, evaluasi)
    base, _ = prediksi_baseline(latih_baseline(train), evaluasi)
    preds = {'baseline_tanpa_kalibrasi': base, 'lgb_sebelum': pred, 'lgb_sesudah': terapkan_koreksi(pred, delta)}
    args.output_dir.mkdir(parents=True, exist_ok=True)
    keseluruhan = pd.DataFrame([{'model': nama, 'jumlah': len(evaluasi), **metrik(evaluasi.durasi_jam.to_numpy(), p)} for nama, p in preds.items()])
    keseluruhan.to_csv(args.output_dir / 'metrik_keseluruhan.csv', index=False)
    tabel = {}
    for kolom in DIMENSI:
        tabel[kolom] = pd.concat([tabel_kelompok(evaluasi, p, kolom, nama) for nama, p in preds.items()], ignore_index=True)
        tabel[kolom].to_csv(args.output_dir / f'error_per_{kolom}.csv', index=False)
        ringkas = tabel[kolom][tabel[kolom].model == 'lgb_sesudah']
        print(f'\n{kolom}: tiga kelompok dengan pinball P90 tertinggi (setelah kalibrasi)')
        print(ringkas.sort_values('pinball_p90', ascending=False).head(3)[['kelompok', 'jumlah', 'pinball_p50', 'pinball_p90', 'coverage_p90', 'durasi_di_atas_720_jam']].to_string(index=False))
    diagnostik_ekstrem(evaluasi, preds['lgb_sesudah']).to_csv(args.output_dir / 'diagnostik_target_ekstrem.csv', index=False)
    gambar_coverage(tabel, args.output_dir / 'coverage_per_kelompok.png')
    meta = {'python': platform.python_version(), 'versi': {p: version(p) for p in ['numpy', 'pandas', 'lightgbm', 'matplotlib']}, 'pembagian': pembagian, 'koreksi_p90_jam': delta, 'metode': 'Model dan koreksi identik dengan kalibrasi_regresi.py; tidak tuning dari hasil kelompok Agustus.', 'batasan': 'Metrik deskriptif, bukan sebab-akibat atau uji signifikansi. Kelompok kecil lebih tidak stabil. Durasi ekstrem dipertahankan; ambang 720 jam hanya diagnostik. Dimensi dianalisis terpisah, bukan kombinasi layanan-pulau-kurir. Agustus sudah dilihat sebelumnya; evaluasi historis. Batasan kematangan label dan pemilihan hyperparameter dari tahap kalibrasi tetap berlaku. Tidak membuktikan alasan model; SHAP/PDP masih tahap berikutnya.'}
    (args.output_dir / 'konfigurasi.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
    print('\nLolos: metrik kelompok berbobot jumlah paket sama dengan metrik keseluruhan.')
    print('Hasil dan grafik:', args.output_dir)

In [ ]:
jalankan_analisis_error_regresi()

## 12. Interpretasi
Lihat gain fitur dan PDP. Ini hubungan prediktif, bukan sebab-akibat.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Interpretasi LightGBM tetap dengan gain dan partial dependence (PDP)."""
import json
import platform
from importlib.metadata import version
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
SEED_interpretasi_regresi = 42

def prediksi_fitur(models, x, koreksi):
    pred = np.column_stack([m.predict(x) for m in models])
    pred[:, 0] = np.maximum(pred[:, 0], 0)
    pred[:, 1] = np.maximum(pred[:, 1], pred[:, 0])
    pred = terapkan_koreksi(pred, koreksi)
    if not np.isfinite(pred).all():
        raise ValueError('Prediksi PDP tidak finite.')
    return pred

def pdp_brute(models, referensi, kolom, grid, koreksi):
    """Rata-rata prediksi ketika satu fitur diganti, bukan kuantil target populasi."""
    hasil = []
    for nilai in grid:
        x = referensi.copy()
        if isinstance(x[kolom].dtype, pd.CategoricalDtype):
            if nilai not in x[kolom].cat.categories:
                raise ValueError('Grid kategorikal harus berasal dari kategori train.')
            x[kolom] = pd.Categorical([nilai] * len(x), categories=x[kolom].cat.categories)
        else:
            x[kolom] = nilai
        pred = prediksi_fitur(models, x, koreksi)
        hasil.append({'fitur': kolom, 'nilai': nilai, 'rata_prediksi_p50_jam': float(pred[:, 0].mean()), 'rata_prediksi_p90_jam': float(pred[:, 1].mean())})
    return pd.DataFrame(hasil)

def grid_fitur(x_train, kolom):
    if kolom in KATEGORI_model_lightgbm_regresi:
        return x_train[kolom].value_counts().head(12).index.tolist()
    nilai = x_train[kolom].dropna().to_numpy()
    unik = np.unique(nilai)
    if not len(unik):
        raise ValueError(f'Fitur seluruhnya kosong: {kolom}')
    if len(unik) <= 20:
        return unik.tolist()
    grid = np.linspace(*np.quantile(nilai, [0.05, 0.95]), 20)
    if pd.api.types.is_integer_dtype(x_train[kolom].dtype):
        grid = np.round(grid)
    return np.unique(grid).tolist()

def grafik_pdp(tabel, pilihan, path):
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), layout='constrained')
    for ax, kolom in zip(axes.flat, pilihan):
        t = tabel[tabel.fitur == kolom]
        if kolom in KATEGORI_model_lightgbm_regresi:
            posisi = np.arange(len(t))
            ax.bar(posisi - 0.18, t.rata_prediksi_p50_jam, width=0.36, color='#087f8c', label='P50')
            ax.bar(posisi + 0.18, t.rata_prediksi_p90_jam, width=0.36, color='#d97706', label='P90 setelah kalibrasi')
            ax.set_xticks(posisi, t.nilai.astype(str), rotation=35, ha='right', fontsize=8)
        else:
            ax.plot(t.nilai.astype(float), t.rata_prediksi_p50_jam, color='#087f8c', label='P50', linewidth=2)
            ax.plot(t.nilai.astype(float), t.rata_prediksi_p90_jam, color='#d97706', label='P90 setelah kalibrasi', linewidth=2)
        ax.set_title(kolom)
        ax.set_ylabel('Rata-rata prediksi (jam)')
        ax.grid(alpha=0.2)
        ax.set_axisbelow(True)
    for ax in list(axes.flat)[len(pilihan):]:
        ax.set_visible(False)
    fig.legend(*axes.flat[0].get_legend_handles_labels(), loc='outside lower center', ncol=2)
    fig.suptitle('Partial dependence: 5 fitur teratas menurut gain gabungan\nModel tetap; referensi sampel train; bukan hubungan sebab-akibat', fontsize=14)
    fig.savefig(path, dpi=150)
    plt.close(fig)

def jalankan_interpretasi_regresi():
    repo = (REPO / 'kode/notebook.py').parents[1]
    args = SimpleNamespace(data_dir=DATA_DIR, output_dir=repo / 'kode/hasil/interpretasi_regresi', sample_size=2000)
    if args.sample_size < 1:
        print('sample-size harus positif.')
    data = pd.read_csv(args.data_dir / 'regresi_train.csv.gz')
    tr, ca, _, pembagian = bagi_kalibrasi(data)
    train, kalibrasi = (data.iloc[tr], data.iloc[ca])
    models, kamus = latih_pasangan(train)
    pred_cal, _, _ = prediksi_pasangan(models, kamus, kalibrasi)
    koreksi, _ = hitung_koreksi(kalibrasi.durasi_jam.to_numpy(), pred_cal[:, 1])
    x_train, _, _ = fitur(train, kamus)
    x_ref = x_train.sample(n=min(args.sample_size, len(x_train)), random_state=SEED_interpretasi_regresi)
    p_ref, _, _ = prediksi_pasangan(models, kamus, train.loc[x_ref.index])
    np.testing.assert_allclose(prediksi_fitur(models, x_ref, koreksi), terapkan_koreksi(p_ref, koreksi))
    gain = pd.DataFrame({'fitur': x_train.columns})
    for model, nama in zip(models, ['p50', 'p90']):
        nilai = model.booster_.feature_importance(importance_type='gain')
        gain[f'gain_{nama}'] = nilai
        gain[f'porsi_gain_{nama}'] = nilai / nilai.sum() if nilai.sum() else 0.0
    gain['porsi_gain_rata'] = (gain.porsi_gain_p50 + gain.porsi_gain_p90) / 2
    gain = gain.sort_values(['porsi_gain_rata', 'fitur'], ascending=[False, True])
    pilihan = gain.head(5).fitur.tolist()
    tables, grid_meta, korelasi_rows = ([], {}, [])
    korelasi = x_train.select_dtypes(include='number').corr(method='spearman')
    for kolom in pilihan:
        grid = grid_fitur(x_train, kolom)
        print(f'PDP {kolom}: {len(grid)} nilai, {len(x_ref):,} baris referensi...', flush=True)
        tables.append(pdp_brute(models, x_ref, kolom, grid, koreksi))
        grid_meta[kolom] = {'jumlah_titik': len(grid), 'kategori': kolom in KATEGORI_model_lightgbm_regresi}
        if kolom in korelasi:
            relasi = korelasi[kolom].drop(kolom).dropna()
            for pasangan in relasi.abs().sort_values(ascending=False).head(3).index:
                korelasi_rows.append({'fitur': kolom, 'fitur_terkait': pasangan, 'spearman_train': float(relasi[pasangan])})
    tabel = pd.concat(tables, ignore_index=True)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    gain.to_csv(args.output_dir / 'importance_gain.csv', index=False)
    tabel.to_csv(args.output_dir / 'partial_dependence.csv', index=False)
    pd.DataFrame(korelasi_rows, columns=['fitur', 'fitur_terkait', 'spearman_train']).to_csv(args.output_dir / 'korelasi_fitur_terpilih.csv', index=False)
    grafik_pdp(tabel, pilihan, args.output_dir / 'partial_dependence.png')
    meta = {'python': platform.python_version(), 'versi': {p: version(p) for p in ['numpy', 'pandas', 'lightgbm', 'matplotlib']}, 'sample_size': len(x_ref), 'seed': SEED_interpretasi_regresi, 'referensi': 'sampel train sebelum Juli', 'pembagian': pembagian, 'fitur_terpilih': pilihan, 'grid': grid_meta, 'koreksi_p90_jam': koreksi, 'metode': 'Brute PDP atas pipeline kuantil setelah koreksi urutan dan kalibrasi.', 'pemilihan_fitur': 'Lima gain tertinggi, rata-rata porsi gain masing-masing model P50/P90.', 'grid_numerik': '20 titik persentil 5-95 train; fitur diskrit kecil memakai semua nilai unik.', 'grid_kategori': 'Maksimal 12 kategori teratas frekuensi train.', 'batasan': 'PDP bukan kausal. Fitur berkorelasi dapat menghasilkan kombinasi hipotetis tidak realistis. Rata-rata menutupi variasi antar paket; bukan kuantil distribusi target gabungan. Gain bukan SHAP dan bukan ukuran akurasi atau bukti leakage. Fitur anonim tidak diberi makna bisnis rekaan. Tidak memakai target Agustus untuk memilih fitur, model, grid, atau koreksi. Tidak mengubah model atau menyelesaikan masalah coverage K07.', 'referensi_metode': 'https://scikit-learn.org/stable/modules/partial_dependence.html'}
    (args.output_dir / 'konfigurasi.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
    print('\nLima fitur teratas (gain bukan persentase pengaruh sebab-akibat):')
    print(gain.head(5)[['fitur', 'porsi_gain_p50', 'porsi_gain_p90', 'porsi_gain_rata']].to_string(index=False))
    print('\nTabel dan grafik tersimpan di:', args.output_dir)
    print('Tidak tuning ulang, tidak membuat prediksi test, dan tidak mengubah dataset mentah.')

In [ ]:
jalankan_interpretasi_regresi()

## 13. Prediksi final
Latih sebelum Agustus, kalibrasi ulang pada label Agustus yang tersedia sebelum September, lalu prediksi test. Kalibrasi final berbeda dari evaluasi historis; skor test belum diketahui.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Model final tetap: fit sebelum Agustus, kalibrasi Agustus, test Sep-Okt."""
import hashlib
import json
from importlib.metadata import version
from pathlib import Path
import numpy as np
import pandas as pd

def periksa_output(output, id_test):
    if list(output.columns) != ['id_paket', 'p50', 'p90']:
        raise ValueError('Kolom harus tepat id_paket,p50,p90.')
    ids = pd.Series(id_test).reset_index(drop=True).astype('string')
    actual = output.id_paket.reset_index(drop=True).astype('string')
    if ids.isna().any() or ids.duplicated().any() or (not actual.equals(ids)):
        raise ValueError('ID harus lengkap, unik, dan urutannya sama dengan test.')
    values = output[['p50', 'p90']].to_numpy(dtype=float)
    if not len(values) or not np.isfinite(values).all() or (values < 0).any():
        raise ValueError('Prediksi harus finite, nonnegatif, dan tidak kosong.')
    if (values[:, 1] < values[:, 0]).any():
        raise ValueError('P90 tidak boleh kurang dari P50.')

def bagi_final(data):
    tr, agustus, info = buat_fold(data, '2025-08-01', '2025-09-01')
    pickup = pd.to_datetime(data.tgl_pickup) + pd.to_timedelta(data.jam_pickup, unit='h')
    tiba = pickup + pd.to_timedelta(data.durasi_jam, unit='h')
    ca = agustus[(tiba.iloc[agustus] < pd.Timestamp('2025-09-01')).to_numpy()]
    assert not np.intersect1d(tr, ca).size
    if not len(ca):
        raise ValueError('Kalibrasi final kosong.')
    return (tr, ca, {**info, 'kalibrasi_agustus': len(ca), 'agustus_belum_tiba_sebelum_september': len(agustus) - len(ca)})

def jalankan_prediksi_final_regresi():
    repo = (REPO / 'kode/notebook.py').parents[1]
    args = SimpleNamespace(data_dir=DATA_DIR, output=repo / 'prediksi_regresi.csv', hasil_dir=repo / 'kode/hasil/prediksi_final_regresi')
    train_path, test_path = (args.data_dir / n for n in ['regresi_train.csv.gz', 'regresi_test.csv.gz'])
    data = pd.read_csv(train_path)
    test = pd.read_csv(test_path, dtype={'id_paket': 'string'})
    tanggal = pd.to_datetime(test.tgl_pickup, errors='raise')
    if tanggal.isna().any() or not ((tanggal >= '2025-09-01') & (tanggal < '2025-11-01')).all():
        raise ValueError('Tanggal test di luar Sep-Okt 2025.')
    if 'durasi_jam' in test:
        raise ValueError('Dataset test seharusnya tidak memiliki label.')
    if set(data.id_paket) & set(test.id_paket):
        raise ValueError('Ada ID train/test bertumpang tindih.')
    tr, ca, pembagian = bagi_final(data)
    train, kalibrasi = (data.iloc[tr], data.iloc[ca])
    models, kamus = latih_pasangan(train)
    p_cal, _, _ = prediksi_pasangan(models, kamus, kalibrasi)
    delta, k = hitung_koreksi(kalibrasi.durasi_jam.to_numpy(), p_cal[:, 1])
    pred, crossing, unknown = prediksi_pasangan(models, kamus, test)
    pred = terapkan_koreksi(pred, delta)
    output = pd.DataFrame({'id_paket': test.id_paket, 'p50': pred[:, 0], 'p90': pred[:, 1]})
    periksa_output(output, test.id_paket)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    output.to_csv(args.output, index=False)
    periksa_output(pd.read_csv(args.output, dtype={'id_paket': 'string'}), test.id_paket)
    args.hasil_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([{'kondisi': nama, **metrik(kalibrasi.durasi_jam.to_numpy(), p)} for nama, p in [('sebelum', p_cal), ('sesudah', terapkan_koreksi(p_cal, delta))]]).to_csv(args.hasil_dir / 'diagnostik_kalibrasi.csv', index=False)
    metadata = {'parameter': {**PARAMETER, **KANDIDAT['lgb_31']}, 'pembagian': pembagian, 'versi': {p: version(p) for p in ['numpy', 'pandas', 'lightgbm', 'scikit-learn']}, 'koreksi_p90_jam': delta, 'urutan_koreksi': k, 'baris_test': len(test), 'unknown_test_per_fitur': unknown, 'crossing_sebelum_koreksi': crossing, 'sha256_input': {p.name: hashlib.file_digest(p.open('rb'), 'sha256').hexdigest() for p in [train_path, test_path]}, 'sha256_output': hashlib.sha256(args.output.read_bytes()).hexdigest(), 'fit': 'Sebelum Agustus, label tersedia sebelum 1 Agustus.', 'kalibrasi': 'Pickup Agustus, label tersedia sebelum 1 September.', 'prediksi': 'Sep-Okt memakai model tetap; tidak refit setelah kalibrasi.', 'batasan': 'Menyisihkan Agustus untuk kalibrasi sehingga tidak fit pada seluruh train. Ini pilihan untuk menjaga koreksi sesuai model, berbeda dari saran refit seluruh train pada panduan. Coverage test tidak diketahui karena label tersembunyi. Diagnostik Agustus bukan evaluasi independen. Asumsi waktu tiba, seleksi label dan drift tetap membatasi coverage. Diagnosis historis dua bulan Juli-Agustus tersedia, tetapi bersifat eksploratif dan tidak menjamin hasil test.'}
    (args.hasil_dir / 'konfigurasi.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print('\nPembagian final:', pembagian)
    print(f'Koreksi P90 baru: {delta:+.6f} jam')
    print('Baris test dengan kategori baru:', unknown)
    print(f'Lolos pemeriksaan CSV: {len(output):,} baris; ID lengkap/unik; nilai finite; P90 >= P50.')
    print('Output:', args.output)
    print('Belum mengetahui skor test; belum commit atau push.')

In [ ]:
jalankan_prediksi_final_regresi()

## 13b. Diagnosis historis horizon dua bulan
Latih sebelum Juni, kalibrasi Juni yang matang sebelum Juli, lalu bekukan model dan koreksi untuk Juli–Agustus. Parameter tetap lgb_31. Periode dan pemilihan parameter sudah pernah diperiksa: hasil eksploratif, bukan holdout baru.


In [ ]:
def bagi_dua_bulan(data):
    tr, juni, info = buat_fold(data, '2025-06-01', '2025-07-01')
    matang_juli, ev, _ = buat_fold(data, '2025-07-01', '2025-09-01')
    ca = np.intersect1d(juni, matang_juli)
    assert len(ca) > 0
    for a, b in [(tr, ca), (tr, ev), (ca, ev)]:
        assert not np.intersect1d(a, b).size
    return tr, ca, ev, {**info, 'kalibrasi_juni': len(ca),
        'juni_label_belum_tersedia': len(juni)-len(ca), 'evaluasi_juli_agustus': len(ev), 'fit_cutoff': '2025-06-01', 'kalibrasi_cutoff': '2025-07-01', 'evaluasi_selesai_eksklusif': '2025-09-01'}


def jalankan_validasi_dua_bulan():
    data = pd.read_csv(DATA_DIR / 'regresi_train.csv.gz')
    tr, ca, ev, info = bagi_dua_bulan(data)
    train, cal, val = data.iloc[tr], data.iloc[ca], data.iloc[ev]
    models, kamus = latih_pasangan(train)
    pcal, _, _ = prediksi_pasangan(models, kamus, cal)
    delta, urutan = hitung_koreksi(cal.durasi_jam.to_numpy(), pcal[:, 1])
    fingerprint_sebelum = [hashlib.sha256(m.booster_.model_to_string().encode()).hexdigest() for m in models]
    before, _, unknown = prediksi_pasangan(models, kamus, val)
    assert fingerprint_sebelum == [hashlib.sha256(m.booster_.model_to_string().encode()).hexdigest() for m in models]
    after = terapkan_koreksi(before, delta)
    base, _ = prediksi_baseline(latih_baseline(train), val)
    months = pd.to_datetime(val.tgl_pickup).dt.month.to_numpy()
    assert set(months) == {7, 8}
    rows = []
    for periode, mask in [('juli', months == 7), ('agustus', months == 8), ('gabungan', np.ones(len(val), dtype=bool))]:
        for model, pred in [('baseline', base), ('lgb_sebelum', before), ('lgb_sesudah', after)]:
            rows.append({'periode': periode, 'model': model, 'jumlah': int(mask.sum()),
                         **metrik(val.durasi_jam.to_numpy()[mask], pred[mask])})
    result = pd.DataFrame(rows)
    for model in result.model.unique():
        subset = result[(result.model == model) & (result.periode != 'gabungan')]
        total = result[(result.model == model) & (result.periode == 'gabungan')].iloc[0]
        for metric in ['pinball_p50', 'pinball_p90', 'coverage_p90']:
            assert np.isclose(np.average(subset[metric], weights=subset.jumlah), total[metric])
    out = REPO / 'kode/hasil/validasi_dua_bulan'
    out.mkdir(parents=True, exist_ok=True)
    result.to_csv(out / 'metrik.csv', index=False)
    pd.DataFrame([info]).to_csv(out / 'pembagian.csv', index=False)
    meta = {'parameter': {**PARAMETER, **KANDIDAT['lgb_31']}, 'koreksi_p90_jam': delta,
            'urutan_koreksi': urutan, 'kategori_baru': unknown, 'sha256_model': fingerprint_sebelum, 'refit_selama_evaluasi': False,
            'fit': 'Pickup sebelum Juni; label tersedia sebelum 1 Juni.',
            'kalibrasi': 'Pickup Juni; label tersedia sebelum 1 Juli.',
            'evaluasi': 'Seluruh label pickup Juli-Agustus; model dan koreksi tetap selama dua bulan.',
            'eksploratif': True,
            'batasan': 'Parameter sudah dipilih lewat eksperimen Mei-Juni dan periode Juli-Agustus pernah diperiksa. '
                       'Ini diagnosis horizon dua bulan secara retrospektif, bukan evaluasi independen atau backtest operasional sepenuhnya. '
                       'Tidak memilih ulang model, fitur, koreksi, atau prediksi final dari hasil ini. '
                       'Asumsi waktu tiba dan seleksi label matang membatasi jaminan coverage.'}
    (out / 'konfigurasi.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
    print(result.to_string(index=False))
    print('Eksploratif: model dan koreksi tetap untuk Juli-Agustus; prediksi final tidak diubah.')

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for nama, group in result[result.periode != 'gabungan'].groupby('model', sort=False):
        for ax, metric in zip(axes, ['coverage_p90', 'pinball_p90']):
            ax.plot(group.periode, group[metric], marker='o', label=nama)
    axes[0].axhline(.9, linestyle='--', color='black', label='Target 90%')
    axes[0].set(ylabel='Coverage P90', ylim=(0, 1), title='Cakupan tiap bulan')
    axes[1].set(ylabel='Pinball loss P90 (jam)', title='Loss tiap bulan')
    axes[0].legend(fontsize=7)
    fig.suptitle('Diagnosis eksploratif: model tetap Juli–Agustus')
    fig.tight_layout()
    fig.savefig(out / 'horizon_dua_bulan.png', dpi=150)
    plt.close(fig)


In [ ]:
jalankan_validasi_dua_bulan()


## 14. Draf laporan
Gabungkan tabel hasil menjadi draf Markdown. Masih perlu digabung dengan Task 2/3 untuk laporan kelompok.

Cell pertama menjelaskan fungsi perhitungan; cell berikutnya menjalankannya.

In [ ]:
"""Menyusun draf bagian laporan dari artefak hasil; tidak melatih model."""
import json
from pathlib import Path
import pandas as pd

def tabel(frame):

    def teks(v):
        if isinstance(v, float):
            return f'{v:.6f}'
        return str(v).replace('|', '\\|').replace('\n', ' ')
    return '\n'.join(['| ' + ' | '.join(map(str, frame.columns)) + ' |', '| ' + ' | '.join(['---'] * len(frame.columns)) + ' |'] + ['| ' + ' | '.join((teks(v) for v in row)) + ' |' for row in frame.itertuples(index=False, name=None)])

def jalankan_susun_laporan_task1():
    kode = (REPO / 'kode/notebook.py').parent
    hasil = kode / 'hasil'

    def csv(path):
        return pd.read_csv(hasil / path)

    def js(path):
        return json.loads((hasil / path).read_text(encoding='utf-8'))
    audit = js('audit_regresi/ringkasan.json')
    lgb = csv('lightgbm_regresi/metrik_gabungan.csv')
    ridge = csv('linear_regresi/metrik_gabungan.csv')
    models = pd.concat([lgb, ridge[ridge.model != 'baseline']], ignore_index=True)
    cols = ['model', 'jumlah_validasi', 'pinball_p50', 'pinball_p90', 'coverage_p90', 'skor_pemilihan']
    kalibrasi = csv('kalibrasi_regresi/metrik.csv')
    final = js('prediksi_final_regresi/konfigurasi.json')
    eval_meta = js('kalibrasi_regresi/konfigurasi.json')
    interpretasi = js('interpretasi_regresi/konfigurasi.json')
    ketahanan = csv('uji_ketahanan_regresi/metrik_gabungan.csv')
    top = []
    for dimensi in ['layanan', 'pulau_tujuan', 'mitra_kurir']:
        g = csv(f'analisis_error_regresi/error_per_{dimensi}.csv')
        g = g[g.model == 'lgb_sesudah'].nlargest(3, 'pinball_p90').copy()
        g.insert(0, 'dimensi', dimensi)
        top.append(g[['dimensi', 'kelompok', 'jumlah', 'pinball_p50', 'pinball_p90', 'coverage_p90']])
    sections = ['# Task 1: Estimasi waktu pengiriman paket', 'Draf bagian laporan untuk digabung dengan Task 2/3. Angka dihasilkan otomatis dari artefak eksperimen; belum merupakan laporan.pdf final kelompok atau klaim persetujuan anggota.', '## Tujuan dan data', f'Memprediksi durasi_jam sebagai P50 dan batas atas P90 pada saat pickup. Tersedia {audit['train']['baris']:,} paket train dan {audit['test']['baris']:,} paket test. Train mencakup Januari–Agustus 2025, test September–Oktober 2025. Data eksternal tidak digunakan. ID hanya untuk identitas; dataset mentah tidak dikumpulkan ke GitHub.', '## Audit kualitas dan ketersediaan fitur', tabel(pd.DataFrame([{'temuan': 'berat kosong train', 'jumlah': audit['train']['nilai_kosong']['berat_kg']}, {'temuan': 'berat kosong test', 'jumlah': audit['test']['nilai_kosong']['berat_kg']}, {'temuan': 'durasi train >720 jam', 'jumlah': audit['target']['jumlah_di_atas_jam']['720']}, {'temuan': 'perkiraan tiba melewati snapshot', 'jumlah': audit['konsistensi_snapshot']['perkiraan_tiba_setelah_snapshot']}, {'temuan': 'test dengan mitra baru', 'jumlah': audit['kategori_test_baru_setelah_normalisasi']['mitra_kurir']['baris_test_terdampak']}])), 'Kehilangan berat tidak seragam antar layanan. LightGBM menangani nilai kosong dengan indikator berat_kosong; Ridge mengisi median dari periode fit saja. Nilai ekstrem tidak otomatis dibuang. Kamus menyebut snapshot 31 Agustus, tetapi sebagian perkiraan tiba melewatinya. Waktu tiba dihitung sebagai pickup + durasi dan diasumsikan label langsung tersedia pada saat itu; jeda pencatatan sebenarnya tidak tersedia. Penyaringan hanya label yang sudah tiba dapat mengurangi representasi paket lambat dalam fit/kalibrasi. Semua label evaluasi tetap dipakai, termasuk paket yang tiba setelah akhir periode, sebagai penilaian retrospektif.', 'Jumlah_scan dikecualikan karena waktu ketersediaannya belum jelas. Fitur f01–f50 dipertahankan karena kamus menyatakan dihitung saat pickup. Hujan_asal_mm dan hujan_tujuan_mm adalah prakiraan hari itu, bukan realisasi hujan sesudah pengiriman. Antrean hub dan indeks kemacetan juga dinyatakan tersedia saat pickup. ID tidak menjadi fitur dan durasi_jam hanya menjadi target. Korelasi atau gain tinggi saja bukan bukti leakage. Eksperimen berikut memakai baris dan parameter yang sama, hanya menambah jumlah_scan:', tabel(csv('audit_leakage_regresi/dampak_scan.csv')), 'Jika scan mencakup kejadian setelah pickup, manfaat tersebut memakai informasi masa depan. Namun eksperimen ini tidak memastikan mekanisme pencatatan; keputusan konservatif tetap tanpa scan. Sumber: [ringkasan audit](audit_regresi/ringkasan.json) dan [eksperimen scan](audit_leakage_regresi/metrik_per_fold.csv).', '## Validasi dan pemilihan model', 'Baseline eksplorasi diuji dengan expanding window Juni–Agustus. Pada setiap cutoff, train hanya memakai pickup sebelumnya dan label yang diperkirakan sudah tersedia. Semua label periode validasi tetap dievaluasi. Perbandingan dengan KFold acak 3 fold, seed 42:', tabel(csv('perbandingan_validasi/metrik_gabungan.csv')), 'Skor acak dan waktu memakai periode evaluasi, ukuran train, serta aturan kematangan label yang berbeda. Selisih tidak mengisolasi efek pengacakan. Skema waktu dipakai karena penggunaan nyata memprediksi bulan berikutnya. Tidak mengklaim skor acak pasti lebih baik untuk semua model.', 'Parameter model lanjutan dipilih pada Mei–Juni. Juli untuk kalibrasi; Agustus untuk evaluasi historis model tetap. Agustus sudah dilihat pada audit/baseline, sehingga bukan holdout yang sama sekali belum pernah dilihat. Pemilihan parameter sebelumnya memakai label Mei–Juni yang diberikan, termasuk sebagian yang mungkin belum tersedia pada cutoff Juli; ini bukan backtest operasional yang sepenuhnya bebas informasi masa depan.', 'Baseline menggunakan kuantil per layanan × pulau dengan fallback kuantil global train. LightGBM quantile memakai 200 pohon, learning rate 0,05, min_child_samples 100, reg_lambda 1, seed 42; num_leaves 15 dan 31 dibandingkan. Ridge memakai log1p target, alpha 1 dan 100, imputasi median, standardisasi dan one-hot hanya dari fit. Bulan terakhir sebelum validasi disisihkan untuk kuantil residual Ridge, tanpa refit sesudahnya. Karena itu ukuran fit Ridge lebih kecil; perbandingan merupakan perbandingan pipeline, bukan efek algoritme murni.', tabel(models[cols]), 'Kriteria pemilihan adalah rata-rata pinball P50/P90 berbobot jumlah baris. Skor ini adalah skor pengembangan, bukan evaluasi independen. LightGBM 31 daun dipilih; dua konfigurasi LightGBM dihitung sebagai satu keluarga model, dan Ridge sebagai keluarga kedua.', '## Ketahanan dan kategori baru', 'Encoder LightGBM belajar kategori dari train; unknown menjadi missing. Ridge menggabungkan kategori jarang (<50) dan menangani unknown melalui OneHotEncoder. Pada simulasi K01/K10 dikeluarkan dari train, semua paket validasi tetap diprediksi. Pengurangan data train dan perubahan komposisi juga memengaruhi hasil; satu simulasi tidak mewakili semua kategori baru.', tabel(ketahanan[ketahanan.subset.isin(['semua', 'kurir_disembunyikan'])][['kondisi', 'subset', 'jumlah_validasi', 'pinball_p50', 'pinball_p90', 'coverage_p90']]), 'Cap target persentil 99 train memperburuk kriteria gabungan, sehingga target asli tetap dipakai setelah penyaringan kematangan label. Target validasi tidak dipotong. Sumber: [audit perlakuan](uji_ketahanan_regresi/audit_perlakuan.csv).', '## Kalibrasi dan evaluasi historis Agustus', 'Skor kalibrasi adalah y−P90. Koreksi memakai statistik urut ke-ceil((n+1)×0,9), kemudian P90 dikoreksi dengan max(P90+koreksi,P50). P50 tidak dikalibrasi. Ini koreksi satu sisi bergaya split conformal; drift waktu dan seleksi label matang membatasi asumsi exchangeability, sehingga coverage 90% tidak dijamin.', f'Koreksi dari Juli: {eval_meta['koreksi_p90_jam']:+.6f} jam. Model tidak di-refit setelah koreksi.', tabel(kalibrasi[kalibrasi.periode == 'agustus_evaluasi'].drop(columns='periode')), 'Baseline pada tabel ini juga memakai train sebelum Juli. Skor diagnostik Juli tidak dipresentasikan sebagai evaluasi independen. Parameter/koreksi tidak dipilih ulang berdasarkan Agustus.', '## Analisis kesalahan per kelompok', tabel(pd.concat(top, ignore_index=True)), 'Peringkat di atas berdasarkan rata-rata pinball P90, bukan jumlah kesalahan total. Dimensi dinilai terpisah. Coverage keseluruhan dapat menyembunyikan masalah suatu kelompok. K07 masih memiliki coverage P90 sekitar 65,78% sesudah koreksi. Ini berarti sekitar sepertiga paket K07 melampaui P90 meskipun cakupan global mendekati 90%. Koreksi global belum memadai untuk kelompok ini. Kalibrasi per kurir dapat diuji pada periode pengembangan baru dengan sampel memadai; tidak diterapkan berdasarkan evaluasi Agustus agar tidak memilih koreksi dari hasil evaluasi. Temuan deskriptif ini tidak membuktikan penyebab operasional maupun perbedaan signifikan secara statistik.', '![Coverage per kelompok](analisis_error_regresi/coverage_per_kelompok.png)', tabel(csv('analisis_error_regresi/diagnostik_target_ekstrem.csv')), 'Ambang 720 jam hanya untuk dekomposisi loss; seluruh 34.168 label Agustus tetap dinilai.', '## Interpretasi partial dependence', 'Lima fitur dipilih dari rata-rata porsi gain P50/P90: ' + ', '.join(interpretasi['fitur_terpilih']) + '. Brute PDP memakai 2.000 baris train, seed 42. Grid numerik menggunakan persentil 5–95 (atau nilai unik jika sedikit); kategori hanya maksimal 12 teratas frekuensi train. Prediksi yang dijelaskan sudah melalui koreksi urutan kuantil dan kalibrasi P90.', '![Partial dependence](interpretasi_regresi/partial_dependence.png)', 'Prediksi ekspres lebih pendek daripada reguler/hemat/kargo pada referensi yang sama; durasi umumnya meningkat seiring jarak pada rentang grid. F04/F17 tetap disebut fitur anonim. PDP bukan kausal dan bukan kuantil populasi; fitur berkorelasi dapat membentuk kombinasi hipotetis tidak realistis. Gain bukan SHAP atau bukti akurasi test. Lihat [catatan interpretasi](../catatan_interpretasi_regresi.md).', '## Prediksi final dan keterbatasan', f'Model final memakai {final['pembagian']['train_dipakai']:,} baris fit sebelum Agustus dan {final['pembagian']['kalibrasi_agustus']:,} label Agustus matang untuk kalibrasi. Koreksi baru {final['koreksi_p90_jam']:+.6f} jam. Model tetap untuk Sep–Okt tanpa refit setelah kalibrasi. Output berisi {final['baris_test']:,} baris dengan kolom id_paket,p50,p90.', 'Agustus kini digunakan sebagai kalibrasi model final; skor evaluasi Agustus di atas berasal dari model historis sebelumnya. Sebagian data disisihkan untuk kalibrasi di luar sampel fit agar residual tidak dihitung dari data yang dipakai melatih model. Refit sesudah kalibrasi akan mengubah model dan membuat koreksi lama tidak lagi sesuai. Ini tradeoff jumlah data fit dengan pemisahan kalibrasi. Koreksi model lama tidak disalin ke model baru. Diagnosis dua bulan Juli–Agustus tersedia sebagai analisis eksploratif; skor/coverage test tetap tidak diketahui. Prediksi diperiksa ID lengkap/unik, urutan sesuai test, finite, nonnegatif, serta P90≥P50 sebelum dan setelah serialisasi CSV.', '## Reproduksi dan kontribusi', 'Dari root repository: `python kode/run_all.py --task 1 --data-dir "LOKASI_DATA"`. Gunakan Python 3.13.0 dan requirements-task1.txt. Seed 42 digunakan untuk prosedur acak. Peluncur menyimpan status dan fingerprint notebook dalam hasil/reproduksi_task1; metadata eksperimen menyimpan parameter dan versi. Instal requirements-notebook.txt untuk kernel Jupyter. Task 2/3 belum diintegrasikan. Untuk ujian lisan, jalankan cell baseline pada kode/Task1_Regresi.ipynb setelah cell definisi sebelumnya.', 'Codex membantu menulis/debug skrip, menjalankan eksperimen, menyusun tabel/grafik dan draf. Validasi mencakup perhitungan manual, tes temporal, kategori baru, kesesuaian metrik agregat, dan pemeriksaan CSV. Anggota harus meninjau serta memahami kode dan mencatat bantuan AI secara jujur pada kontribusi.md; PR pekerjaan ini: https://github.com/telkom-ml-kelompok-11-2026/UTS-ML-KELOMPOK-11/pull/2; persetujuan anggota masih diperlukan.', '## Referensi metode', '- [LightGBM: categorical features](https://lightgbm.readthedocs.io/en/stable/Advanced-Topics.html)\n- [scikit-learn: OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)\n- [scikit-learn: partial dependence](https://scikit-learn.org/stable/modules/partial_dependence.html)\n- [Angelopoulos dan Bates: pengantar conformal prediction](https://arxiv.org/abs/2107.07511)']
    horizon = csv('validasi_dua_bulan/metrik.csv')
    horizon_meta = js('validasi_dua_bulan/konfigurasi.json')
    index = sections.index('## Analisis kesalahan per kelompok')
    sections[index:index] = ['## Diagnosis eksploratif horizon dua bulan',
        'Model dilatih sebelum Juni dengan label yang matang; Juni matang sebelum Juli untuk kalibrasi. Model dan koreksi dibekukan untuk seluruh Juli–Agustus. Parameter lgb_31 tetap; hasil ini tidak digunakan untuk tuning ulang atau mengubah prediksi test.',
        tabel(csv('validasi_dua_bulan/pembagian.csv')), tabel(horizon),
        '![Horizon dua bulan](validasi_dua_bulan/horizon_dua_bulan.png)',
        'Koreksi P90: ' + format(horizon_meta['koreksi_p90_jam'], '+.6f') + ' jam. ' + horizon_meta['batasan']]
    hm = horizon[horizon.model == 'lgb_sesudah'].set_index('periode')
    sections.insert(sections.index('## Analisis kesalahan per kelompok'),
        f"Coverage P90 model terkoreksi: Juli {hm.loc['juli', 'coverage_p90']:.2%}, "
        f"Agustus {hm.loc['agustus', 'coverage_p90']:.2%}, gabungan {hm.loc['gabungan', 'coverage_p90']:.2%}. "
        'Metrik gabungan berbobot jumlah paket, bukan rata-rata sederhana dua bulan. '
        'Perubahan antarbulan menunjukkan ketahanan model tetap; bukan estimasi pasti coverage September–Oktober.')
    before_h = horizon[(horizon.model == 'lgb_sebelum') & (horizon.periode == 'gabungan')].iloc[0]
    after_h = hm.loc['gabungan']
    sections.insert(sections.index('## Analisis kesalahan per kelompok'),
        f"Pada diagnosis ini, coverage gabungan sebelum koreksi {before_h['coverage_p90']:.2%} "
        f"dan sesudah koreksi {after_h['coverage_p90']:.2%}; pinball P90 berubah dari "
        f"{before_h['pinball_p90']:.6f} menjadi {after_h['pinball_p90']:.6f}. "
        'Koreksi Juni bernilai negatif sehingga menurunkan batas P90, dan hasil dua bulan ini justru memburuk. '
        'Kalibrasi historis tidak otomatis memperbaiki periode berikutnya; drift dan seleksi label matang merupakan keterbatasan yang mungkin berperan, bukan penyebab yang sudah dibuktikan. '
        'Coverage masih di bawah sasaran 90%. Hasil ini dilaporkan apa adanya dan tidak dipakai untuk mengubah koreksi atau prediksi final secara retrospektif.')
    output = hasil / 'laporan_task1.md'
    output.write_text('\n\n'.join(sections) + '\n', encoding='utf-8')
    print('Draf laporan dari hasil eksperimen:', output)
    print('Belum laporan.pdf kelompok; perlu penyuntingan bersama dan pengaturan batas halaman.')

In [ ]:
jalankan_susun_laporan_task1()

## 15. Pemeriksaan otomatis
Tes kecil memeriksa aturan pembagian data, kalibrasi, prediksi, dan interpretasi. Tidak melatih ulang model.

In [ ]:
"""Verifikasi metrik kelompok dan kontribusi error dari contoh kecil."""
import unittest
import numpy as np
import pandas as pd

class AnalisisChecks(unittest.TestCase):

    def test_metrik_manual_dan_kelompok_kosong(self):
        data = pd.DataFrame({'layanan': ['A', 'A', 'B', None], 'durasi_jam': [10.0, 30.0, 50.0, 40.0]}, index=[8, 2, 90, 15])
        pred = np.array([[20, 25], [20, 25], [40, 60], [40, 40]])
        tabel = tabel_kelompok(data, pred, 'layanan', 'contoh').set_index('kelompok')
        self.assertEqual(tabel.loc['A', 'jumlah'], 2)
        self.assertEqual(tabel.loc['A', 'pinball_p50'], 5)
        self.assertEqual(tabel.loc['A', 'pinball_p90'], 3)
        self.assertEqual(tabel.loc['A', 'coverage_p90'], 0.5)
        self.assertEqual(tabel.loc['(kosong)', 'jumlah'], 1)
        self.assertAlmostEqual(tabel.kontribusi_loss_p90_persen.sum(), 100)

    def test_subset_ekstrem_tetap_mencakup_semua(self):
        data = pd.DataFrame({'durasi_jam': [100.0, 720.0, 800.0]})
        pred = np.array([[90.0, 110.0], [700.0, 730.0], [500.0, 600.0]])
        tabel = diagnostik_ekstrem(data, pred).set_index('subset')
        self.assertEqual(tabel.loc['durasi_sampai_720_jam', 'jumlah'], 2)
        self.assertEqual(tabel.loc['durasi_di_atas_720_jam', 'jumlah'], 1)
        self.assertAlmostEqual(tabel.kontribusi_loss_p90_persen.sum(), 100)
        self.assertAlmostEqual(tabel.kontribusi_error_absolut_p50_persen.sum(), 100)

    def test_prediksi_tidak_valid_ditolak(self):
        data = pd.DataFrame({'layanan': ['A'], 'durasi_jam': [10.0]})
        with self.assertRaises(ValueError):
            tabel_kelompok(data, np.array([[1.0, np.nan]]), 'layanan', 'contoh')
        with self.assertRaises(ValueError):
            tabel_kelompok(data, [], 'layanan', 'contoh')

In [ ]:
"""Verifikasi PDP terhadap perhitungan manual dan tipe kategori."""
import unittest
import numpy as np
import pandas as pd

class ModelLinearContoh:

    def __init__(self, tambahan):
        self.tambahan = tambahan

    def predict(self, x):
        return 2 * x['x'].to_numpy() + x['z'].to_numpy() + self.tambahan

class InterpretasiChecks(unittest.TestCase):

    def test_pdp_manual_dan_sumber_tidak_berubah(self):
        ref = pd.DataFrame({'x': [1.0, 3.0], 'z': [4.0, 8.0]})
        awal = ref.copy(deep=True)
        t = pdp_brute([ModelLinearContoh(0), ModelLinearContoh(10)], ref, 'x', [2.0, 5.0], 4.0)
        np.testing.assert_allclose(t.rata_prediksi_p50_jam, [10, 16])
        np.testing.assert_allclose(t.rata_prediksi_p90_jam, [24, 30])
        pd.testing.assert_frame_equal(ref, awal)

    def test_grid_train_diskrit_dan_rentang(self):
        train = pd.DataFrame({'x': np.arange(101, dtype=float)})
        grid = grid_fitur(train, 'x')
        self.assertEqual(len(grid), 20)
        self.assertEqual(grid[0], 5)
        self.assertEqual(grid[-1], 95)
        self.assertEqual(grid_fitur(pd.DataFrame({'x': [1, 2, 2]}), 'x'), [1, 2])

    def test_kategori_dipertahankan_dan_unknown_ditolak(self):

        class ModelKategori:

            def predict(self, x):
                assert isinstance(x.layanan.dtype, pd.CategoricalDtype)
                return np.where(x.layanan == 'a', 10.0, 20.0)
        ref = pd.DataFrame({'layanan': pd.Categorical(['a', 'b'], categories=['a', 'b'])})
        t = pdp_brute([ModelKategori(), ModelKategori()], ref, 'layanan', ['a', 'b'], 2.0)
        np.testing.assert_allclose(t.rata_prediksi_p50_jam, [10, 20])
        np.testing.assert_allclose(t.rata_prediksi_p90_jam, [12, 22])
        with self.assertRaises(ValueError):
            pdp_brute([ModelKategori(), ModelKategori()], ref, 'layanan', ['baru'], 2.0)

In [ ]:
"""Uji batas waktu dan perhitungan koreksi P90 dengan contoh manual."""
import unittest
import numpy as np
import pandas as pd

class KalibrasiChecks(unittest.TestCase):

    def test_koreksi_memakai_statistik_urut(self):
        delta, k = hitung_koreksi(np.arange(19.0), np.zeros(19))
        self.assertEqual(k, 18)
        self.assertEqual(delta, 17.0)
        negatif, _ = hitung_koreksi(np.arange(19.0), np.full(19, 19.0))
        self.assertEqual(negatif, -2.0)

    def test_koreksi_tidak_mengubah_p50_dan_sumber(self):
        pred = np.array([[10.0, 12.0], [20.0, 35.0]])
        awal = pred.copy()
        hasil = terapkan_koreksi(pred, -5.0)
        np.testing.assert_array_equal(hasil, [[10, 10], [20, 30]])
        np.testing.assert_array_equal(pred, awal)
        np.testing.assert_array_equal(terapkan_koreksi(pred, 2), [[10, 14], [20, 37]])

    def test_input_tidak_valid(self):
        with self.assertRaises(ValueError):
            hitung_koreksi([], [])
        with self.assertRaises(ValueError):
            hitung_koreksi([1], [0])
        with self.assertRaises(ValueError):
            hitung_koreksi(np.ones(20), np.full(20, np.nan))

    def test_kematangan_label_dan_evaluasi_utuh(self):
        d = pd.DataFrame({'tgl_pickup': ['2025-06-01', '2025-06-30', '2025-07-01', '2025-07-31', '2025-08-01', '2025-08-31'], 'jam_pickup': [0] * 6, 'durasi_jam': [24, 72, 24, 48, 24, 720]})
        tr, ca, ev, info = bagi_kalibrasi(d)
        self.assertEqual(tr.tolist(), [0])
        self.assertEqual(ca.tolist(), [2])
        self.assertEqual(ev.tolist(), [4, 5])
        self.assertEqual(info['juli_belum_tiba_sebelum_agustus'], 1)
        d.loc[ev, 'durasi_jam'] = 99999
        tr2, ca2, ev2, _ = bagi_kalibrasi(d)
        np.testing.assert_array_equal(tr, tr2)
        np.testing.assert_array_equal(ca, ca2)
        np.testing.assert_array_equal(ev, ev2)

In [ ]:
"""Pemeriksaan batas waktu, kategori baru, dan transformasi kuantil."""
import unittest
import numpy as np
import pandas as pd

class LinearChecks(unittest.TestCase):

    def test_label_dan_periode_terpisah(self):
        d = pd.DataFrame({'tgl_pickup': ['2025-01-01', '2025-03-31', '2025-04-01', '2025-04-30', '2025-05-01', '2025-05-31'], 'jam_pickup': [0] * 6, 'durasi_jam': [24, 48, 24, 72, 24, 24]})
        fit, res, _, val, info = bagi_internal(d, '2025-05-01', '2025-06-01')
        self.assertEqual(fit.tolist(), [0])
        self.assertEqual(res.tolist(), [2])
        self.assertEqual(val.tolist(), [4, 5])
        self.assertEqual(info['residual_belum_matang'], 1)

    def test_transformasi_kuantil_manual(self):

        class Model:

            def predict(self, x):
                return np.full(len(x), np.log(3))
        pred = prediksi_kuantil(Model(), [0, 1], np.array([0, np.log(2)]))
        np.testing.assert_allclose(pred, [[2, 5], [2, 5]])

    def test_unknown_missing_dan_fitur_terlarang(self):
        d = pd.DataFrame({col: np.arange(120, dtype=float) + 1 for col in NUMERIK})
        d['tgl_pickup'] = '2025-01-01'
        for col in KATEGORI_model_lightgbm_regresi:
            d[col] = ['a'] * 100 + ['jarang'] * 20
        d.loc[0, 'berat_kg'] = np.nan
        x = fitur_linear(d)
        model = pipeline_linear(1)
        model.fit(x, np.log1p(np.arange(120) + 10))
        med = model.named_steps['preproses'].named_transformers_['numerik'].named_steps['isi_kosong'].statistics_.copy()
        v = d.iloc[:2].copy()
        for col in KATEGORI_model_lightgbm_regresi:
            v[col] = 'kategori_baru'
        pred = model.predict(fitur_linear(v))
        self.assertTrue(np.isfinite(pred).all())
        v['jumlah_scan'], v['durasi_jam'], v['id_paket'] = (999999, -999, 'id_baru')
        np.testing.assert_allclose(model.predict(fitur_linear(v)), pred)
        np.testing.assert_array_equal(med, model.named_steps['preproses'].named_transformers_['numerik'].named_steps['isi_kosong'].statistics_)
        self.assertFalse({'jumlah_scan', 'durasi_jam', 'id_paket'} & set(fitur_linear(v)))

In [ ]:
import unittest
import numpy as np
import pandas as pd

class FinalChecks(unittest.TestCase):

    def test_output_valid(self):
        periksa_output(pd.DataFrame({'id_paket': ['P1', 'P2'], 'p50': [1, 2], 'p90': [1, 3]}), ['P1', 'P2'])

    def test_output_rusak_ditolak(self):
        dasar = pd.DataFrame({'id_paket': ['P1', 'P2'], 'p50': [1.0, 2.0], 'p90': [1.0, 3.0]})
        for kolom, nilai in [('id_paket', ['P1', 'P1']), ('id_paket', ['P2', 'P1']), ('p50', [np.nan, 2]), ('p90', [0.0, 3.0]), ('p90', [np.inf, 3.0])]:
            d = dasar.copy()
            d[kolom] = nilai
            with self.subTest(kolom=kolom, nilai=nilai), self.assertRaises(ValueError):
                periksa_output(d, ['P1', 'P2'])
        with self.assertRaises(ValueError):
            periksa_output(dasar.iloc[:1], ['P1', 'P2'])
        with self.assertRaises(ValueError):
            periksa_output(dasar.assign(extra=0), ['P1', 'P2'])

    def test_batas_final(self):
        d = pd.DataFrame({'tgl_pickup': ['2025-07-01', '2025-07-31', '2025-08-01', '2025-08-31'], 'jam_pickup': [0] * 4, 'durasi_jam': [24, 48, 24, 48]})
        tr, ca, info = bagi_final(d)
        self.assertEqual(tr.tolist(), [0])
        self.assertEqual(ca.tolist(), [2])
        self.assertEqual(info['agustus_belum_tiba_sebelum_september'], 1)

In [ ]:
"""Periksa cap train dan simulasi kategori baru tanpa training berat."""
import unittest
import numpy as np
import pandas as pd

class KetahananChecks(unittest.TestCase):

    def test_cap_tidak_mengubah_sumber(self):
        train = pd.DataFrame({'durasi_jam': np.arange(1.0, 101.0)})
        sebelum = train.copy(deep=True)
        y, batas, n = target_dibatasi(train)
        self.assertAlmostEqual(batas, 99.01)
        self.assertEqual(n, 1)
        np.testing.assert_array_equal(y.iloc[:99], np.arange(1.0, 100.0))
        self.assertAlmostEqual(y.iloc[-1], 99.01)
        pd.testing.assert_frame_equal(train, sebelum)

    def test_kurir_dipilih_tanpa_target_dan_urutan(self):
        train = pd.DataFrame({'mitra_kurir': ['K01', 'K02', 'K03', 'K04'] * 10, 'durasi_jam': np.arange(40)})
        pilihan = pilih_kurir(train)
        train['durasi_jam'] = 999999
        self.assertEqual(pilihan, pilih_kurir(train.iloc[::-1]))
        self.assertEqual(len(pilihan), 2)

    def test_kategori_disembunyikan_benar_benar_baru(self):
        d = pd.DataFrame({k: np.ones(40) for k in NUMERIK})
        d['tgl_pickup'] = '2025-01-01'
        for k in KATEGORI_model_lightgbm_regresi:
            d[k] = 'kategori'
        d['mitra_kurir'] = ['K01', 'K02', 'K03', 'K04'] * 10
        dipilih = pilih_kurir(d)
        mask = normalisasi_uji_ketahanan_regresi(d.mitra_kurir).isin(dipilih)
        _, kamus, _ = fitur(d.loc[~mask])
        x_val, _, baru = fitur(d.loc[mask], kamus)
        self.assertTrue(x_val.mitra_kurir.isna().all())
        self.assertEqual(baru['mitra_kurir'], int(mask.sum()))
        self.assertFalse(set(dipilih) & set(kamus['mitra_kurir']))
class HorizonChecks(unittest.TestCase):
    def test_batas_waktu_dan_label_lambat(self):
        data = pd.DataFrame({
            'tgl_pickup': ['2025-05-01', '2025-05-31', '2025-06-01', '2025-06-30', '2025-07-01', '2025-08-31', '2025-09-01'],
            'jam_pickup': [0]*7, 'durasi_jam': [24, 48, 24, 48, 24, 1000, 24]})
        tr, ca, ev, _ = bagi_dua_bulan(data)
        np.testing.assert_array_equal(tr, [0])
        np.testing.assert_array_equal(ca, [2])
        np.testing.assert_array_equal(ev, [4, 5])

    def test_model_tetap_dua_bulan(self):
        from unittest.mock import patch
        import tempfile
        data = pd.DataFrame({'tgl_pickup': ['2025-05-01']*12 + ['2025-06-01']*12 + ['2025-07-01']*3 + ['2025-08-01']*4,
                             'jam_pickup': [0]*31, 'durasi_jam': [24]*31})
        model = SimpleNamespace(booster_=SimpleNamespace(model_to_string=lambda: 'model-tetap'))
        calls = []
        def fit(frame):
            calls.append(frame.index.tolist())
            return [model, model], {}
        def predict(models, kamus, frame):
            return np.tile([20., 30.], (len(frame), 1)), 0, {}
        def base(model, frame): return np.tile([20., 30.], (len(frame), 1)), 0
        with tempfile.TemporaryDirectory(dir=REPO / 'kode/hasil') as tmp:
            with patch.dict(globals(), {'REPO': Path(tmp), 'latih_pasangan': fit,
                 'prediksi_pasangan': predict, 'latih_baseline': lambda frame: None, 'prediksi_baseline': base}):
                with patch.object(pd, 'read_csv', return_value=data):
                    jalankan_validasi_dua_bulan()
            metrics = pd.read_csv(Path(tmp) / 'kode/hasil/validasi_dua_bulan/metrik.csv')
        self.assertEqual(calls, [list(range(12))])
        self.assertEqual(metrics.query("periode == 'gabungan' and model == 'lgb_sesudah'").iloc[0]['jumlah'], 7)


In [ ]:
suite = unittest.TestSuite()
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(AnalisisChecks))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(InterpretasiChecks))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(KalibrasiChecks))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(LinearChecks))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(FinalChecks))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(KetahananChecks))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(HorizonChecks))
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
assert result.wasSuccessful(), "Ada tes gagal; periksa pesan di atas."
print(f"Lolos: {result.testsRun} tes; tidak ada kegagalan atau error.")


## 16. Ringkasan hasil
Tabel dan grafik di bawah menggunakan hasil yang dihasilkan cell sebelumnya. Label test tidak tersedia.

In [ ]:
import pandas as pd
display(Markdown("### Perbandingan model pada pengembangan Mei–Juni"))
display(pd.read_csv(REPO / "kode/hasil/lightgbm_regresi/metrik_gabungan.csv"))
display(pd.read_csv(REPO / "kode/hasil/linear_regresi/metrik_gabungan.csv"))
display(Markdown("### Evaluasi historis Agustus"))
evaluation = pd.read_csv(REPO / "kode/hasil/kalibrasi_regresi/metrik.csv")
display(evaluation[evaluation["periode"] == "agustus_evaluasi"])
display(Markdown("### Lima baris pertama prediksi test"))
prediction = pd.read_csv(REPO / "prediksi_regresi.csv")
display(prediction.head())
print("Jumlah baris prediksi:", len(prediction))
display(Image(filename=str(REPO / "kode/hasil/analisis_error_regresi/coverage_per_kelompok.png")))
display(Image(filename=str(REPO / "kode/hasil/interpretasi_regresi/partial_dependence.png")))
print("Draf laporan:", REPO / "kode/hasil/laporan_task1.md")
print("Draf kontribusi:", REPO / "kode/kontribusi_task1_draf.md")